In [ ]:

import matplotlib.pyplot as plt
import time
import torch
import numpy as np
from sympy.codegen.rewriting import Optimization
from tqdm import tqdm
from tqdm.notebook import tqdm
import pandas as pd
import torch
import os
from datetime import datetime
import sys
import json
from matplotlib.ticker import LogLocator, LogFormatter
from tqdm import trange
from functools import partial

In [ ]:
import os
import sys
from huggingface_hub import login
from google.colab import userdata
import wandb

hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass

    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    github_token = userdata.get('GITHUB')
    if github_token:
        token = github_token
    else:
        token = getpass.getpass("Enter your GitHub personal access token: ")

    repo_url  = f"https://{token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "adding-simu-compare"

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"✅ Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

    repo_path = f"/content/{repo_name}"
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    print(f"\n✅ Repo ready. Branch: {branch}")
    print(f"📁 Python path: {repo_path}")

repo_path = f"/content/{repo_name}/simulations"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)



In [ ]:
import importlib
import Diffusion
import DiffusionGeneral
import superposition_utils

import LossFunctions
import ConsistencyModels
import  FlowMatching
from ConsistencyModels import ConsistencyModel,ConsistencyModeliCT
import dist_utils
import Optimization
import EncoderDecoder
import evalModels

importlib.reload(Diffusion)
importlib.reload(LossFunctions)

importlib.reload(ConsistencyModels)
importlib.reload(FlowMatching)
importlib.reload(dist_utils)
importlib.reload(Optimization)
importlib.reload(superposition_utils)
importlib.reload(DiffusionGeneral)
importlib.reload(EncoderDecoder)
importlib.reload(evalModels)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# 2D cond on 1D

In [ ]:
mu_list = [torch.tensor([-5,5],dtype=torch.float64),
           torch.tensor([-5,-5],dtype=torch.float64),
           torch.tensor([5, 3],dtype=torch.float64),
           torch.tensor([5,-1],dtype=torch.float64),
           torch.tensor([0, -3],dtype=torch.float64),
           torch.tensor([-2,4 ],dtype=torch.float64),
           torch.tensor([-2,-3 ],dtype=torch.float64),
           torch.tensor([ 1,2],dtype=torch.float64),
           torch.tensor([-7,1],dtype=torch.float64),
           torch.tensor([7,5],dtype=torch.float64),
           torch.tensor([0,-5],dtype=torch.float64)

           ]

# Convert Sigma_list to 3D covariance matrices
Sigma_list = [
    torch.tensor([[0.5000, 0.1950],
 [0.1950, 0.2000]], dtype=torch.float64)
] * len(mu_list)

# Mixture weights
alpha =torch.tensor( [1 / len(mu_list)] * len(mu_list),dtype=torch.float64)

mu_list = [mu.float() for mu in mu_list]
Sigma_list = [cov.float() for cov in Sigma_list]
alpha = alpha.float()

## Target
x_star=torch.tensor([-5])
mu_temp, Sigma_temp =dist_utils.compute_conditionals(mu_list, Sigma_list, x_star)
temp_alpha = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_star)
mog_means, mog_variances, weights= dist_utils.filter_and_normalize(mu_temp, Sigma_temp, temp_alpha, threshold=0.01)


In [ ]:
X = dist_utils.generate_mog_samples(25_000, mu_list, Sigma_list, alpha).float().to(device)
# Convert to numpy
xh_cpu = X.detach().cpu().numpy()

# Scatter plot of first column vs. second column
plt.scatter(xh_cpu[:, 0], xh_cpu[:, 1], alpha=0.6, s=20)
plt.title("Scatter Plot of First vs. Second Columns of xh")
plt.xlabel("First Column")
plt.ylabel("Second Column")
plt.grid(True)
plt.show()

## Parameters for tests

In [ ]:
## NN
nblocks=3
nunits=128
nepochs=20_000
batch_size=512

nepochs_CM=7_500
batch_size_CM=1024
## Diffusion
diffusion_steps=100

## Optimization
n_attemp_optim=25
nsamples_in_optim_for_mmd=250


# For models
condition_on=1
nfeatures= X.shape[1]

## SuperPosition

### Train

In [ ]:
point_l = [[int(p[1]), int(p[0])] for p in mu_list]
mu_list_super = [torch.tensor(p, dtype=torch.float64) for p in point_l]
Sigma_list_super = [s[torch.tensor([1, 0])][:, torch.tensor([1, 0])] for s in Sigma_list]# [.T for sigma in Sigma_list]

mu_list_super = [mu.float() for mu in mu_list_super]
Sigma_list_super = [cov.float() for cov in Sigma_list_super]
alpha = alpha.float()

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list_super, variances=Sigma_list_super, weights=alpha,kernel_func=None)
model_cond_diff =DiffusionGeneral.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=True,condition_on=condition_on,diffusion_steps=diffusion_steps)
model_cond_diff.train_model(
    data_generator=data_generator,#partial(dist_utils.generate_mog_samples, xi=mu_list_super, S=Sigma_list_super, beta=alpha, kernel_func=None),
    dataloader=None, num_iterations=nepochs)


### Optimize

In [ ]:
n_sample=n_attemp_optim
cond1= torch.tensor([5.],device=device)
cond2= torch.tensor([-5.],device=device)


cond_l= [cond1,cond2]



start_time = time.time()
x_gen_iso,_=  superposition_utils.generate_isosurface_samples(model_cond_diff, cond_l=cond_l,bs=n_sample,device=device)
end_time = time.time()
superposition_times = [end_time - start_time] *n_attemp_optim

X_gen_np = x_gen_iso[:, -1, :].detach().cpu().numpy()




In [ ]:
# X_gen_np
plt.hist(X_gen_np[:,1], bins=50,range=(-10,10))  # You can adjust number of bins
plt.show()

In [ ]:
best_x_t_superposition_list=[]
l1_distance_superposition_list=[]
norm_swd_superposition_list=[]
swd_superposition_list=[]

for optimal_x in  X_gen_np[:,1]:

    l1_distance = dist_utils.warpper_L1_distance(torch.tensor([optimal_x]),mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)

    swd,norm_swd = LossFunctions.compute_swd(
        best_x0_sample=torch.tensor([optimal_x]),
        mu_list=mu_list,
        Sigma_list=Sigma_list,
        alpha=alpha,
        mog_means=mog_means,
        mog_variances=mog_variances,
        weights=weights,
        decoder=None,
        scaler=None,
        device="cpu",
        nsamples=10_000,
        num_projections=1_000,
        p=2,
        Flag=False
    )
    print(l1_distance)
    print(norm_swd)


    l1_distance_superposition_list.append(l1_distance )
    norm_swd_superposition_list.append(norm_swd)
    swd_superposition_list.append(swd)
    best_x_t_superposition_list.append(torch.tensor([optimal_x]))

print(best_x_t_superposition_list)
print(l1_distance_superposition_list)
print(norm_swd_superposition_list)

plt.hist(l1_distance_superposition_list,  edgecolor='black',range=(0,2),bins=100,)
plt.show()

plt.hist(norm_swd_superposition_list, edgecolor='black', range=(0, max(norm_swd_superposition_list)), bins=100)

plt.show()

## Consistency Models

In [ ]:
B, C = X.shape
nfeatures = C - condition_on

# Use the converted data
data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(nfeatures=nfeatures, condition_on=condition_on, nunits=128)
Cos_ConsistencyModeliCT.train_model(
    X=None,
    nepochs=nepochs_CM,
    batch_size=batch_size_CM,
    device=device,
    condition=condition_on,
    data_generator=data_generator,
    use_improved_training=True
)

## LGD

### Train

In [ ]:
## LGD
# init a model, train
X = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)
nfeatures = X.shape[1]
condition_on = mu_list[0].shape[0] - mog_means[0].shape[0]
model_cond = Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=True,
                                      condition_on=condition_on, diffusion_steps=diffusion_steps)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
losses = model_cond.train_model(None,
                                data_generator=data_generator,
                                nepochs=nepochs, batch_size=batch_size, condition_on=condition_on)

In [ ]:
# init a model, train
X_for_cond_only = X[:, :model_cond.condition_on]
nfeatures = X_for_cond_only.shape[1]

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :model_cond.condition_on])
model_uncond=Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=False,diffusion_steps=diffusion_steps)
losses = model_uncond.train_model(None,
                                  data_generator=data_generator,
                                  nepochs=nepochs
, batch_size=batch_size, condition_on=condition_on)


### Optimize

#### Regular

In [ ]:
best_x_t_LGD_list=[]
l1_distance_LGD_list=[]
norm_swd_LGD_list=[]
swd_LGD_list=[]

lgd_times = []
final_loss_LGD=[]

for i in trange(n_attemp_optim):
    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(model_uncond, model_cond, mog_means, mog_variances, weights, mu_list,
                                                                           Sigma_list, alpha, nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device)
    end_time = time.time()
    lgd_times.append(end_time - start_time)

    best_x_t_LGD_list.append(best_x_t)

    l1_distance = dist_utils.warpper_L1_distance(best_x_t, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
    swd,norm_swd = LossFunctions.compute_swd(
        best_x0_sample=best_x_t,
        mu_list=mu_list,
        Sigma_list=Sigma_list,
        alpha=alpha,
        mog_means=mog_means,
        mog_variances=mog_variances,
        weights=weights,
        decoder=None,
        scaler=None,
        device="cpu",
        nsamples=10_000,
        num_projections=1_000,
        p=2,
        Flag=False
    )

    print(l1_distance)
    print(norm_swd)
    l1_distance_LGD_list.append(l1_distance)
    norm_swd_LGD_list.append(norm_swd)
    swd_LGD_list.append(swd)
    final_loss_LGD.append(final_loss)

print(best_x_t_LGD_list)
print(l1_distance_LGD_list)
print(norm_swd_LGD_list)

plt.hist(l1_distance_LGD_list,  edgecolor='black',range=(0,2),bins=100,)
plt.show()

plt.hist(norm_swd_LGD_list, edgecolor='black', range=(0, max(norm_swd_LGD_list)), bins=100)
plt.show()

### Optimize

#### CM

In [ ]:
best_x_t_LGD_CM_list=[]
l1_distance_LGD_CM_list=[]
norm_swd_LGD_CM_list=[]
swd_LGD_CM_list=[]

lgd_cm_times = []
final_loss_LGD_CM=[]

for i in trange(n_attemp_optim):
    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(model_uncond, Cos_ConsistencyModeliCT, mog_means, mog_variances, weights, mu_list, Sigma_list, alpha, nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device, CM=True, FLAG=False,num_x_t=3)
    end_time = time.time()
    lgd_cm_times.append(end_time - start_time)
    best_x_t_LGD_CM_list.append(best_x_t)

    l1_distance = dist_utils.warpper_L1_distance(best_x_t, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
    # EMD_distance = dist_utils.compute_mixture_wasserstein_distance_1D(mu_list, Sigma_list, alpha, x_star, best_x_t,x_range=(-5, 15), n_points=1000)["wasserstein_distance"]
    swd,norm_swd =  LossFunctions.compute_swd(
        best_x0_sample=best_x_t,
        mu_list=mu_list,
        Sigma_list=Sigma_list,
        alpha=alpha,
        mog_means=mog_means,
        mog_variances=mog_variances,
        weights=weights,
        decoder=None,
        scaler=None,
        device="cpu",
        nsamples=10_000,
        num_projections=1_000,
        p=2,
        Flag=False
    )

    print(l1_distance)
    print(norm_swd)

    l1_distance_LGD_CM_list.append(l1_distance)
    norm_swd_LGD_CM_list.append(norm_swd)
    swd_LGD_CM_list.append(swd)
    final_loss_LGD_CM.append(final_loss)

print(best_x_t_LGD_CM_list)
print(l1_distance_LGD_CM_list)
print(norm_swd_LGD_CM_list)

plt.hist(l1_distance_LGD_CM_list,  edgecolor='black',range=(0,2),bins=100,)
plt.show()

plt.hist(norm_swd_LGD_CM_list, edgecolor='black', range=(0,50), bins=100)
plt.show()

## Flow

### Train

In [ ]:
input_dim =mog_means[0].shape[0]
condition_on = mu_list[0].shape[0]-mog_means[0].shape[0]
hidden_dim = nunits
depth = nblocks

In [ ]:
vf_y_cond_x = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=condition_on,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
vf_y_cond_x.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,data_generator=data_generator)



In [ ]:
input_dim = mu_list[0].shape[0]-mog_means[0].shape[0]#first_column_x.shape[1]
vf_X = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=0,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)
data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :condition_on])

vf_X.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,
              data_generator=data_generator
              )



In [ ]:
x_optim, loss = Optimization.optimize_DFLOW(vf_y_cond_x, vf_X, device, mog_means, mog_variances, weights, max_iter=100, FLAG=True, n_sample=nsamples_in_optim_for_mmd, loss_method="MMD", line_search_fn="strong_wolfe")

### Optimize

#### Regular

In [ ]:
l1_distance_dflow_list=[]
x_optim_dflow_list=[]
dflow_times = []
final_loss_dflow_list=[]
norm_swd_dflow_list=[]
swd_dflow_list=[]

for i in trange(n_attemp_optim):
    start_time = time.time()
    x_optim, final_loss = Optimization.optimize_DFLOW(vf_y_cond_x, vf_X, device, mog_means, mog_variances, weights, max_iter=100, FLAG=False, n_sample=nsamples_in_optim_for_mmd, loss_method="MMD", line_search_fn="strong_wolfe")
    end_time = time.time()
    dflow_times.append(end_time - start_time)

    x_optim_dflow_list.append(x_optim)

    l1_distance = dist_utils.warpper_L1_distance(x_optim, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
    # EMD_distance = dist_utils.compute_mixture_wasserstein_distance_1D(mu_list, Sigma_list, alpha, x_star, x_optim,x_range=(-5, 15), n_points=1000)["wasserstein_distance"]
    swd,norm_swd = LossFunctions.compute_swd(
        best_x0_sample=x_optim,
        mu_list=mu_list,
        Sigma_list=Sigma_list,
        alpha=alpha,
        mog_means=mog_means,
        mog_variances=mog_variances,
        weights=weights,
        decoder=None,
        scaler=None,
        device="cpu",
        nsamples=10_000,
        num_projections=1_000,
        p=2,
        Flag=False
    )



    l1_distance_dflow_list.append(l1_distance)
    x_optim_dflow_list.append(x_optim)
    norm_swd_dflow_list.append(norm_swd)
    swd_dflow_list.append(swd)
    final_loss_dflow_list.append(final_loss)

l1_distance_dflow_list = [tensor.detach().cpu().numpy() if hasattr(tensor, 'detach') else tensor for tensor in l1_distance_dflow_list]


print(l1_distance_dflow_list)
print(x_optim_dflow_list)

plt.hist(l1_distance_dflow_list,  edgecolor='black',range=(0,2),bins=100,)
plt.show()

plt.hist(norm_swd_dflow_list, edgecolor='black', range=(0, max(norm_swd_dflow_list)), bins=100)
plt.show()


In [ ]:
# Convert tensor list to Python list of values
tensor_string ="[tensor([[-4.6943]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.6943]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.1526]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.1526]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.7446]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.7446]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.3974]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.3974]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.8506]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.8506]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-2.1662]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-2.1662]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.3702]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.3702]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.5939]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.5939]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[0.8542]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[0.8542]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[6.2200]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[6.2200]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.2046]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.2046]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.4788]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.4788]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.1864]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.1864]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.6755]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.6755]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.1824]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.1824]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.5975]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.5975]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[0.4512]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[0.4512]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.6889]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.6889]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[0.5884]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[0.5884]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[4.7296]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[4.7296]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-1.2278]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-1.2278]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.1912]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.1912]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.8585]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.8585]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.8478]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.8478]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[6.2032]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[6.2032]], device='cuda:0', grad_fn=<SelectBackward0>)]"
# CHANGED: Convert tensors to list of scalar values
import re
values_list = [float(x) for x in re.findall(r'\[\[([-\d.]+)\]\]', tensor_string)]
converted_list = [torch.tensor([[val]], device='cpu') for val in values_list]

# converted_list = [float(x) for x in re.findall(r'\[\[([-\d.]+)\]\]', tensor_string)]

# Initialize list to store results
norm_swd_dflow_list = []

# CHANGED: Loop over converted_list instead of n_attemp_optim
# for i in trange(n_attemp_optim):
for i, x_optim in enumerate(trange(len(converted_list))):
    swd, norm_swd = LossFunctions.compute_swd(
        # CHANGED: Use value from converted_list
        # best_x0_sample=x_optim,
        best_x0_sample=converted_list[i],
        mu_list=mu_list,
        Sigma_list=Sigma_list,
        alpha=alpha,
        mog_means=mog_means,
        mog_variances=mog_variances,
        weights=weights,
        decoder=None,
        scaler=None,
        device="cpu",
        nsamples=10_000,
        num_projections=1_000,
        p=2,
        Flag=False
    )

    norm_swd_dflow_list.append(norm_swd)


In [ ]:
norm_swd_dflow_list

## Summary - to 10

In [ ]:
# Prepare data dictionaries
norm_swd_dict = {
    'Superposition': norm_swd_superposition_list,
    'LGD': norm_swd_LGD_list,
    'LGD-CM': norm_swd_LGD_CM_list,
    'D-FLOW': norm_swd_dflow_list
}

swd_dict = {
    'Superposition': swd_superposition_list,
    'LGD': swd_LGD_list,
    'LGD-CM': swd_LGD_CM_list,
    'D-FLOW': swd_dflow_list
}

l1_dict = {
    'Superposition': l1_distance_superposition_list,
    'LGD': l1_distance_LGD_list,
    'LGD-CM': l1_distance_LGD_CM_list,
    'D-FLOW': l1_distance_dflow_list
}

times_dict = {
    'Superposition': [0] * len(norm_swd_superposition_list),
    'LGD': lgd_times,
    'LGD-CM': lgd_cm_times,
    'D-FLOW': dflow_times
}

In [ ]:



# TOP 10 indices
top10_indices = {
    'Superposition': np.random.choice(len(norm_swd_superposition_list), 10, replace=False),
    'LGD': np.argsort(evalModels.to_numpy(final_loss_LGD))[:10],
    'LGD-CM': np.argsort(evalModels.to_numpy(final_loss_LGD_CM))[:10],
    'D-FLOW': np.argsort(evalModels.to_numpy(final_loss_dflow_list))[:10]
}

# ALL indices
all_indices = {
    'Superposition': list(range(len(norm_swd_superposition_list))),
    'LGD': list(range(len(norm_swd_LGD_list))),
    'LGD-CM': list(range(len(norm_swd_LGD_CM_list))),
    'D-FLOW': list(range(len(norm_swd_dflow_list)))
}

# Generate tables
df_top10, summary_top10 = evalModels.create_summary_table(
    top10_indices, norm_swd_dict, swd_dict, l1_dict, times_dict,
    title="TOP 10 - Metrics Summary"
)

df_all, summary_all =evalModels.create_summary_table(
    all_indices, norm_swd_dict, swd_dict, l1_dict, times_dict,
    title="ALL ATTEMPTS - Metrics Summary"
)

In [ ]:
# Usage example:
models_config = {
    'Diffusion_Conditional': {
        'model': model_cond,
        'type': 'diffusion',
        'task': 'P(Y|X)',
        'extract_generated_only': True,
        'reverse_distribution': False
    },
    'FlowMatching_Conditional': {
        'model': vf_y_cond_x,
        'type': 'fm',
        'task': 'P(Y|X)',
        'extract_generated_only': False,
        'reverse_distribution': False
    },
    'ConsistencyModel': {
        'model': Cos_ConsistencyModeliCT,
        'type': 'cm',
        'task': 'P(Y|X)',
        'extract_generated_only': False,
        'reverse_distribution': False
    },
    'SuperDiffusion': {
        'model': model_cond_diff,
        'type': 'super_diffusion',
        'task': 'P(X|Y)',
        'extract_generated_only': True,
        'reverse_distribution': True
    },
    'Diffusion_Unconditional': {
        'model': model_uncond,
        'type': 'diffusion',
        'task': 'P(X)',
        'extract_generated_only': False,
        'reverse_distribution': False
    },
    'FlowMatching_Unconditional': {
        'model': vf_X,
        'type': 'fm',
        'task': 'P(X)',
        'extract_generated_only': False,
        'reverse_distribution': False
    },
}


# Run evaluation
df_results, summary_stats = evalModels.create_model_evaluation_summary(
    models_dict=models_config,
    mu_list=mu_list,
    Sigma_list=Sigma_list,
    alpha=alpha,
    condition_on=condition_on,
    device=device,
    num_evaluations=250,
    nsamples=2_500,
    num_projections=1000
)

In [ ]:
import matplotlib.pyplot as plt

# Get best x_star from each method
best_indices = {
    'LGD': np.argsort(evalModels.to_numpy(final_loss_LGD))[0],
    'LGD-CM': np.argsort(evalModels.to_numpy(final_loss_LGD_CM))[0],
    'D-FLOW': np.argsort(evalModels.to_numpy(final_loss_dflow_list))[0]
}

# Colors for each method
colors = {'LGD': 'red', 'LGD-CM': 'green', 'D-FLOW': 'purple'}

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot theoretical PDF once
x_range = torch.linspace(-10, 10, 1000, device='cuda:0')
pdf_theory = dist_utils.mog_pdf(x_range, mog_means.squeeze(), mog_variances.squeeze(), weights)
ax.plot(x_range.cpu(), pdf_theory.cpu(), 'b-', label='Theoretical', linewidth=5, alpha=0.8)

# Plot each method
for method, best_idx in best_indices.items():
    # Get decoded x_star
    if method == 'LGD':
        decoded_original = best_x_t_LGD_list[best_idx]
    elif method == 'LGD-CM':
        decoded_original = best_x_t_LGD_CM_list[best_idx]
    else:  # D-FLOW
        decoded_original = x_optim_dflow_list[best_idx]

    # Compute conditionals
    mu_cond, Sigma_cond = dist_utils.compute_conditionals(
        mu_list, Sigma_list, decoded_original.reshape(-1, 1)
    )
    alpha_cond = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, decoded_original)

    # Get SWD distance
    swd_norm = norm_swd_dict[method][best_idx]

    # Plot optimized PDF
    pdf_optim = dist_utils.mog_pdf(x_range, mu_cond.squeeze(), Sigma_cond.squeeze(), alpha_cond)
    ax.plot(x_range.cpu(), pdf_optim.detach().cpu().numpy(), '--',
            color=colors[method],
            label=f'{method} (x*={decoded_original.item():.3f}, SWD={swd_norm:.4f})',
            linewidth=2)

ax.set_xlabel('Y', fontsize=12)
ax.set_ylabel('PDF', fontsize=12)
ax.set_title('Conditional Distributions: Theoretical vs Optimized Methods', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# change
# Create boxplot for all four distance lists
data = [l1_distance_dflow_list, l1_distance_superposition_list, l1_distance_LGD_list, l1_distance_LGD_CM_list]

labels = ['DFlow', 'Superposition', 'LGD', 'LGD_CM']

plt.figure(figsize=(12, 6))
plt.boxplot(data, labels=labels)
plt.ylabel('L1 Distance')
plt.title('L1 Distance Comparison - All Methods')
plt.grid(True, alpha=0.3)
plt.show()
# changed

In [ ]:
# Plot optimization times
time_data = [dflow_times, superposition_times, lgd_times, lgd_cm_times]
time_labels = ['DFlow', 'Superposition', 'LGD', 'LGD_CM']

plt.figure(figsize=(12, 6))
plt.boxplot(time_data, labels=time_labels)
plt.ylabel('Optimization Time (seconds)')
plt.title('Optimization Time Comparison')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
stop

# 10D cond on 1D

## Parameters for tests

In [ ]:
## NN
nblocks=5
nunits=128
nepochs=30_000
batch_size=512

nepochs_CM=7_500
batch_size_CM=4096

## Diffusion
diffusion_steps=100

## Optimization
n_attemp_optim=25
nsamples_in_optim_for_mmd=250

In [ ]:
mu_list, Sigma_list, alpha,mog_means, mog_variances,weights,x_star= dist_utils.get_param_mog_with_target(dim_data=10,num_components=4,device='cpu',conditional_modes=2,distanceOrScale="Distance")
mog_means, mog_variances, weights= dist_utils.filter_and_normalize(mog_means, mog_variances, weights, threshold=0.001)


In [ ]:
x_star

## SuperPosition

### Train

In [ ]:
point_l = [p.flip(0) for p in mu_list]
mu_list_super = point_l
idx = torch.arange(Sigma_list[0].size(0) - 1, -1, -1)
Sigma_list_super = [s[idx][:, idx]  for s in Sigma_list]

mu_list_super = [mu.float() for mu in mu_list_super]
Sigma_list_super = [cov.float() for cov in Sigma_list_super]
alpha = alpha.float()

nfeatures= mu_list[0].shape[0]
condition_on=1

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list_super, variances=Sigma_list_super, weights=alpha,kernel_func=None)
model_cond_diff =DiffusionGeneral.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=True,condition_on=condition_on,diffusion_steps=diffusion_steps)
model_cond_diff.train_model(
    data_generator=data_generator,
    dataloader=None, num_iterations=1)


### Optimize

In [ ]:
n_sample=n_attemp_optim
first_value = mog_means[0].item()
second_value = mog_means[1].item()
cond1= torch.tensor([first_value],device=device)
cond2= torch.tensor([second_value],device=device)
cond3= torch.tensor([first_value+0.2],device=device)
cond4= torch.tensor([second_value-0.1],device=device)

start_time = time.time()
cond_l= [cond1,cond2,cond3,cond4]
x_gen_iso,_=  superposition_utils.generate_isosurface_samples(model_cond_diff, cond_l=cond_l,bs=n_sample,device=device)
X_gen_np = x_gen_iso[:, -1, :].detach().cpu().numpy()
end_time = time.time()


superposition_times = [end_time - start_time] *n_attemp_optim


# Flip again
X_gen_np_original = torch.tensor(X_gen_np)[:,condition_on:]
X_gen_np_original=X_gen_np_original.flip(1)


In [ ]:
best_x_t_superposition_list=[]
l1_distance_superposition_list=[]
norm_swd_superposition_list=[]
swd_superposition_list=[]

for i in range(X_gen_np_original.shape[0]):
    val = X_gen_np_original[i:i+1].reshape(-1, 1)
    l1_distance = dist_utils.warpper_L1_distance(val,mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
    swd,norm_swd = LossFunctions.compute_swd(
        best_x0_sample=val,#torch.tensor([optimal_x]),
        mu_list=mu_list,
        Sigma_list=Sigma_list,
        alpha=alpha,
        mog_means=mog_means,
        mog_variances=mog_variances,
        weights=weights,
        decoder=None,
        scaler=None,
        device="cpu",
        nsamples=10_000,
        num_projections=1_000,
        p=2,
        Flag=False
    )


    l1_distance_superposition_list.append(l1_distance )
    norm_swd_superposition_list.append(norm_swd)
    swd_superposition_list.append(swd)

    best_x_t_superposition_list.append(val)

print(best_x_t_superposition_list)
print(l1_distance_superposition_list)
print(norm_swd_superposition_list)

plt.hist(l1_distance_superposition_list,  edgecolor='black',range=(0,2),bins=100,)
plt.show()

plt.hist(norm_swd_superposition_list,  edgecolor='black',range=(0, max(norm_swd_superposition_list)),bins=100,)
plt.show()

In [ ]:
plt.scatter(norm_swd_superposition_list,l1_distance_superposition_list)

In [ ]:
print("swd_superposition_list",swd_superposition_list)

## Consistency Models

In [ ]:
## LGD
# init a model, train
X = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)

condition_on=9
B, C = X.shape
nfeatures = C - condition_on
mu_list = [mu.float() for mu in mu_list]
Sigma_list = [cov.float() for cov in Sigma_list]
alpha = alpha.float()

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(nfeatures=nfeatures, condition_on=condition_on, nunits=256)
Cos_ConsistencyModeliCT.train_model(X=None, nepochs=1
                      ,batch_size=batch_size_CM, device= device, condition=condition_on,
                      data_generator=data_generator,
                      )

## LGD

### Train

In [ ]:
## LGD
# init a model, train
X = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)
nfeatures = X.shape[1]
condition_on = mu_list[0].shape[0] - mog_means[0].shape[0]
model_cond = Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=True,
                                      condition_on=condition_on, diffusion_steps=diffusion_steps)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
losses = model_cond.train_model(None,
                                data_generator=data_generator,
                                nepochs=1, batch_size=batch_size, condition_on=condition_on)

In [ ]:
# init a model, train
X_for_cond_only = X[:, :model_cond.condition_on]
nfeatures = X_for_cond_only.shape[1]

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :model_cond.condition_on])
model_uncond=Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=False,diffusion_steps=diffusion_steps)
losses = model_uncond.train_model(None,
                                  data_generator=data_generator,
                                  nepochs=1
, batch_size=batch_size, condition_on=condition_on)


### Optimize

#### Regular

In [ ]:
best_x_t_LGD_list=[]
l1_distance_LGD_list=[]
EMD_distance_LGD_list=[]
norm_swd_LGD_list=[]

lgd_times = []
final_loss_LGD=[]
swd_LGD_list=[]

for i in trange(n_attemp_optim):
    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(model_uncond, model_cond, mog_means, mog_variances, weights, mu_list, Sigma_list, alpha, nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device)
    best_x_t=best_x_t.reshape(-1,1)
    end_time = time.time()
    lgd_times.append(end_time - start_time)

    best_x_t_LGD_list.append(best_x_t)

    l1_distance = dist_utils.warpper_L1_distance(best_x_t, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
    EMD_distance = dist_utils.compute_mixture_wasserstein_distance_1D(mu_list, Sigma_list, alpha, x_star, best_x_t,x_range=(-5, 15), n_points=1000)["wasserstein_distance"]
    swd,norm_swd = LossFunctions.compute_swd(
          best_x0_sample=best_x_t,
          mu_list=mu_list,
          Sigma_list=Sigma_list,
          alpha=alpha,
          mog_means=mog_means,
          mog_variances=mog_variances,
          weights=weights,
          decoder=None,
          scaler=None,
          device="cpu",
          nsamples=10_000,
          num_projections=1_000,
          p=2,
          Flag=False
      )
    l1_distance_LGD_list.append(l1_distance)
    norm_swd_LGD_list.append(norm_swd)
    swd_LGD_list.append(swd)
    final_loss_LGD.append(final_loss)


print(best_x_t_LGD_list)
print(l1_distance_LGD_list)
print(norm_swd_LGD_list)

plt.hist(l1_distance_LGD_list,  edgecolor='black',range=(0,2),bins=100,)
plt.show()

plt.hist(norm_swd_LGD_list,  edgecolor='black',range=(0, max(norm_swd_LGD_list)),bins=100,)
plt.show()

In [ ]:
print(f"final_loss_LGD:",final_loss_LGD)
print(f"swd_LGD_list:",swd_LGD_list)


### Optimize

#### CM

In [ ]:
best_x_t_LGD_CM_list=[]
l1_distance_LGD_CM_list=[]
norm_swd_LGD_CM_list=[]
lgd_cm_times = []
final_loss_LGD_CM=[]
swd_LGD_CM_list=[]

for i in trange(n_attemp_optim):
    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(model_uncond, Cos_ConsistencyModeliCT, mog_means, mog_variances, weights, mu_list, Sigma_list, alpha, nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device, CM=True, FLAG=False,num_x_t=10)
    end_time = time.time()
    best_x_t=best_x_t.reshape(-1,1)
    lgd_cm_times.append(end_time - start_time)
    best_x_t_LGD_CM_list.append(best_x_t)

    l1_distance = dist_utils.warpper_L1_distance(best_x_t, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
    swd,norm_swd =  LossFunctions.compute_swd(
        best_x0_sample=best_x_t,
        mu_list=mu_list,
        Sigma_list=Sigma_list,
        alpha=alpha,
        mog_means=mog_means,
        mog_variances=mog_variances,
        weights=weights,
        decoder=None,
        scaler=None,
        device="cpu",
        nsamples=10_000,
        num_projections=1_000,
        p=2,
        Flag=False
    )
    print(l1_distance)
    print(norm_swd)
    l1_distance_LGD_CM_list.append(l1_distance)
    norm_swd_LGD_CM_list.append(norm_swd)
    swd_LGD_CM_list.append(swd)
    final_loss_LGD_CM.append(final_loss)

print(best_x_t_LGD_CM_list)
print(l1_distance_LGD_CM_list)
print(norm_swd_LGD_CM_list)

plt.hist(l1_distance_LGD_CM_list,  edgecolor='black',range=(0,2),bins=100,)
plt.show()

plt.hist(norm_swd_LGD_CM_list,  edgecolor='black', range=(0,50),bins=100,)
plt.show()

In [ ]:
print(f"final_loss_LGD_CM:",final_loss_LGD_CM)
print("swd_LGD_CM_list",swd_LGD_CM_list)

## Flow

### Train

In [ ]:
input_dim =mog_means[0].shape[0]
y_dim = mu_list[0].shape[0]-mog_means[0].shape[0]
hidden_dim = nunits
depth = nblocks


In [ ]:
vf_y_cond_x = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=condition_on,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
vf_y_cond_x.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,data_generator=data_generator)



In [ ]:

input_dim = mu_list[0].shape[0]-mog_means[0].shape[0]#first_column_x.shape[1]
vf_X = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=0,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)
data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :condition_on])

vf_X.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,
              data_generator=data_generator
              )

### Optimize

#### Regular

In [ ]:
best_x_t_dflow_list=[]
l1_distance_dflow_list=[]
EMD_distance_dflow_list=[]
dflow_times = []
final_loss_dflow_list=[]
norm_swd_dflow_list=[]
swd_dflow_list=[]

for i in trange(n_attemp_optim):
    start_time = time.time()
    x_optim,  final_loss = Optimization.optimize_DFLOW(vf_y_cond_x, vf_X, device, mog_means, mog_variances, weights, max_iter=100, FLAG=False, n_sample=nsamples_in_optim_for_mmd, loss_method="MMD", line_search_fn="strong_wolfe")
    end_time = time.time()
    x_optim=x_optim.reshape(-1,1)
    dflow_times.append(end_time - start_time)

    best_x_t_dflow_list.append(x_optim)

    l1_distance = dist_utils.warpper_L1_distance(x_optim, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
    EMD_distance = dist_utils.compute_mixture_wasserstein_distance_1D(mu_list, Sigma_list, alpha, x_star, x_optim,x_range=(-5, 15), n_points=1000)["wasserstein_distance"]
    swd,norm_swd = LossFunctions.compute_swd(
            best_x0_sample=x_optim,
            mu_list=mu_list,
            Sigma_list=Sigma_list,
            alpha=alpha,
            mog_means=mog_means,
            mog_variances=mog_variances,
            weights=weights,
            decoder=None,
            scaler=None,
            device="cpu",
            nsamples=10_000,
            num_projections=1_000,
            p=2,
            Flag=False
        )
    l1_distance_dflow_list.append(l1_distance)
    norm_swd_dflow_list.append(norm_swd)
    swd_dflow_list.append(swd)
    final_loss_dflow_list.append(final_loss)

l1_distance_dflow_list = [tensor.detach().cpu().numpy() if hasattr(tensor, 'detach') else tensor for tensor in l1_distance_dflow_list]
# EMD_distance_dflow_list = [tensor.detach().cpu().numpy() if hasattr(tensor, 'detach') else tensor for tensor in EMD_distance_dflow_list]


print(best_x_t_dflow_list)
print(l1_distance_dflow_list)

plt.hist(l1_distance_dflow_list,  edgecolor='black',range=(0,2),bins=100,)
plt.show()

plt.hist(norm_swd_dflow_list,  edgecolor='black',range=(0, max(norm_swd_dflow_list)),bins=100,)
plt.show()


In [ ]:
best_x_t_dflow_list=[]
l1_distance_dflow_list=[]
EMD_distance_dflow_list=[]
dflow_times = []
final_loss_dflow_list=[]
norm_swd_dflow_list=[]
swd_dflow_list=[]

for i in trange(n_attemp_optim):
    start_time = time.time()
    x_optim,  final_loss = Optimization.optimize_DFLOW(vf_y_cond_x, vf_X, device, mog_means, mog_variances, weights, max_iter=100, FLAG=False, n_sample=nsamples_in_optim_for_mmd, loss_method="MMD", line_search_fn="strong_wolfe")
    end_time = time.time()
    x_optim=x_optim.reshape(-1,1)
    dflow_times.append(end_time - start_time)

    best_x_t_dflow_list.append(x_optim)

    l1_distance = dist_utils.warpper_L1_distance(x_optim, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
    EMD_distance = dist_utils.compute_mixture_wasserstein_distance_1D(mu_list, Sigma_list, alpha, x_star, x_optim,x_range=(-5, 15), n_points=1000)["wasserstein_distance"]
    swd,norm_swd = LossFunctions.compute_swd(
            best_x0_sample=x_optim,
            mu_list=mu_list,
            Sigma_list=Sigma_list,
            alpha=alpha,
            mog_means=mog_means,
            mog_variances=mog_variances,
            weights=weights,
            decoder=None,
            scaler=None,
            device="cpu",
            nsamples=10_000,
            num_projections=1_000,
            p=2,
            Flag=False
        )
    l1_distance_dflow_list.append(l1_distance)
    norm_swd_dflow_list.append(norm_swd)
    swd_dflow_list.append(swd)
    final_loss_dflow_list.append(final_loss)

l1_distance_dflow_list = [tensor.detach().cpu().numpy() if hasattr(tensor, 'detach') else tensor for tensor in l1_distance_dflow_list]
# EMD_distance_dflow_list = [tensor.detach().cpu().numpy() if hasattr(tensor, 'detach') else tensor for tensor in EMD_distance_dflow_list]


print(best_x_t_dflow_list)
print(l1_distance_dflow_list)
print(norm_swd_dflow_list)
plt.hist(l1_distance_dflow_list,  edgecolor='black',range=(0,2),bins=100,)
plt.show()

plt.hist(norm_swd_dflow_list,  edgecolor='black',range=(0, max(norm_swd_dflow_list)),bins=100,)
plt.show()


In [ ]:
# Convert tensor list to Python list of values
tensor_string ="""[tensor([[-7.3012],
        [-2.3872],
        [-2.1202],
        [-6.0233],
        [ 1.8390],
        [ 1.7498],
        [-2.4056],
        [-9.0532],
        [-7.7810]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -6.5964],
        [ 18.3037],
        [ -8.0791],
        [-16.4614],
        [  0.3447],
        [ 18.9387],
        [ -3.1774],
        [-10.9091],
        [ -5.7324]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -3.0959],
        [  4.4967],
        [  4.1094],
        [-10.8654],
        [  6.5899],
        [ -8.0759],
        [ -0.4622],
        [ -3.4983],
        [  2.4795]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[-3.3083],
        [ 1.5461],
        [-2.6917],
        [-2.7544],
        [ 1.7054],
        [ 1.9936],
        [-2.2623],
        [ 2.1667],
        [-8.0107]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ 15.6898],
        [ -8.8183],
        [  9.6114],
        [-10.5340],
        [  2.1718],
        [ -0.1897],
        [ -0.8168],
        [ 12.5857],
        [  5.2665]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[-39.9869],
        [ 16.7143],
        [-16.3592],
        [-26.8847],
        [  9.1306],
        [ -1.3870],
        [ -0.7208],
        [-23.9577],
        [-17.1953]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -4.5412],
        [  4.2662],
        [ -5.7762],
        [-14.8041],
        [  6.6922],
        [  5.7406],
        [  5.3337],
        [ -0.8903],
        [  0.2464]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -4.5971],
        [  8.1400],
        [ -0.5944],
        [-14.1688],
        [  4.5927],
        [  2.3237],
        [  2.8799],
        [ -7.0184],
        [  0.6402]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ 15.5432],
        [-14.2056],
        [  8.3671],
        [ -6.6079],
        [  9.5952],
        [ -8.3582],
        [  0.4539],
        [ 11.1425],
        [ -1.6078]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[-1.6631],
        [ 1.7214],
        [-4.3078],
        [-6.2330],
        [ 2.9008],
        [ 2.4351],
        [-3.3391],
        [-3.1519],
        [-5.2522]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -7.1807],
        [  3.1096],
        [  1.5633],
        [-10.5558],
        [  4.1145],
        [  1.4642],
        [  5.9399],
        [  3.1877],
        [ -2.3012]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -3.2306],
        [  6.3263],
        [ -8.0791],
        [-17.0441],
        [  6.0681],
        [ -2.4093],
        [  5.0802],
        [ -3.5548],
        [ -4.2216]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[-4.1160],
        [ 1.6770],
        [ 2.5018],
        [-8.0273],
        [ 1.6119],
        [-1.3880],
        [ 0.1923],
        [ 0.8242],
        [-5.3253]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -4.7856],
        [  0.1789],
        [ -1.8044],
        [-10.5364],
        [  4.7248],
        [ -1.1542],
        [  8.0694],
        [ -4.3458],
        [ -6.0362]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -5.0837],
        [ -0.2603],
        [ -0.4315],
        [-10.3254],
        [  6.2225],
        [  0.8077],
        [  3.7349],
        [ -0.5078],
        [ -0.1202]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[-1.3680],
        [ 0.5028],
        [-5.8757],
        [-0.7374],
        [ 2.4569],
        [ 1.2135],
        [-4.3249],
        [-3.9021],
        [-7.8673]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ 14.0109],
        [-10.6245],
        [  5.9156],
        [ -7.5306],
        [  2.3942],
        [ -2.3407],
        [  3.3719],
        [  9.4920],
        [  2.2870]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -1.8531],
        [  7.9054],
        [  1.5887],
        [-14.4531],
        [  5.8746],
        [ -2.5741],
        [  4.9087],
        [  0.8052],
        [ -3.6284]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[-6.6810],
        [ 1.7758],
        [-6.4990],
        [-6.3405],
        [ 1.6892],
        [-0.7429],
        [-7.5017],
        [-2.3207],
        [-9.1663]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[-2.6165],
        [ 0.7404],
        [-2.1572],
        [-4.6950],
        [ 2.0806],
        [ 1.6862],
        [-0.8336],
        [-3.9320],
        [-7.2940]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[-2.5514],
        [ 2.3314],
        [-6.5051],
        [-4.3512],
        [ 0.9192],
        [-2.8106],
        [-3.9472],
        [-4.1046],
        [-6.9006]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -2.8368],
        [  5.2004],
        [  1.5822],
        [-14.1257],
        [  2.9605],
        [ -3.1216],
        [  9.4066],
        [ -3.8637],
        [ -3.7792]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -3.8384],
        [  2.9879],
        [-10.5666],
        [-20.2770],
        [  2.4524],
        [  2.3613],
        [  3.5463],
        [ -1.7354],
        [  2.8535]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[-8.2760],
        [-3.5464],
        [-2.0458],
        [-0.8624],
        [ 1.8165],
        [ 6.2610],
        [ 0.7934],
        [-4.8605],
        [-6.7535]], device='cuda:0', grad_fn=<ViewBackward0>), tensor([[ -6.3430],
        [  2.7742],
        [ -3.1854],
        [-12.3315],
        [  1.1153],
        [  1.2255],
        [ 11.4007],
        [  1.9957],
        [-10.6977]], device='cuda:0', grad_fn=<ViewBackward0>)]"""


In [ ]:
a=torch.Tensor([[-7.3012],
        [-2.3872],
        [-2.1202],
        [-6.0233],
        [ 1.8390],
        [ 1.7498],
        [-2.4056],
        [-9.0532],
        [-7.7810]], )
a

In [ ]:
swd, norm_swd = LossFunctions.compute_swd(
        best_x0_sample=a.reshape(-1,1),
        mu_list=mu_list,
        Sigma_list=Sigma_list,
        alpha=alpha,
        mog_means=mog_means,
        mog_variances=mog_variances,
        weights=weights,
        decoder=None,
        scaler=None,
        device="cpu",
        nsamples=10_000,
        num_projections=1_000,
        p=2,
        Flag=False
    )

In [ ]:

# Extract all numeric values and group them into tensors
all_values = re.findall(r'\[\s*([-\d.]+)\s*\]', tensor_string)
all_values = [float(x) for x in all_values]

# CHANGED: Create tensors with shape (9,) instead of (9, 1)
# OLD: tensor_2d = torch.tensor([[val] for val in tensor_values], device='cpu')
converted_list = []
for i in range(0, len(all_values), 9):
    tensor_values = all_values[i:i+9]
    tensor_1d = torch.tensor(tensor_values, device='cpu')
    converted_list.append(tensor_1d.reshape(-1,1))

# Initialize list to store results
norm_swd_dflow_list = []

# Loop over converted_list
for i in trange(len(converted_list)):
    swd, norm_swd = LossFunctions.compute_swd(
        best_x0_sample=converted_list[i],
        mu_list=mu_list,
        Sigma_list=Sigma_list,
        alpha=alpha,
        mog_means=mog_means,
        mog_variances=mog_variances,
        weights=weights,
        decoder=None,
        scaler=None,
        device="cpu",
        nsamples=10_000,
        num_projections=1_000,
        p=2,
        Flag=False
    )

    norm_swd_dflow_list.append(norm_swd)

In [ ]:
converted_list

In [ ]:
print("norm_swd_dflow_list",norm_swd_dflow_list)
print("final_loss_dflow_list",final_loss_dflow_list)

## Summary

### Top 10

In [ ]:
# Prepare data dictionaries
norm_swd_dict = {
    'Superposition': norm_swd_superposition_list,
    'LGD': norm_swd_LGD_list,
    'LGD-CM': norm_swd_LGD_CM_list,
    'D-FLOW': norm_swd_dflow_list
}

swd_dict = {
    'Superposition': swd_superposition_list,
    'LGD': swd_LGD_list,
    'LGD-CM': swd_LGD_CM_list,
    'D-FLOW': swd_dflow_list
}

l1_dict = {
    'Superposition': l1_distance_superposition_list,
    'LGD': l1_distance_LGD_list,
    'LGD-CM': l1_distance_LGD_CM_list,
    'D-FLOW': l1_distance_dflow_list
}

times_dict = {
    'Superposition': [0] * len(norm_swd_superposition_list),
    'LGD': lgd_times,
    'LGD-CM': lgd_cm_times,
    'D-FLOW': dflow_times
}

In [ ]:
best_x_t_dflow_list

In [ ]:
import matplotlib.pyplot as plt

# Get best x_star from each method
best_indices = {
    'LGD': np.argsort(evalModels.to_numpy(final_loss_LGD))[0],
    'LGD-CM': np.argsort(evalModels.to_numpy(final_loss_LGD_CM))[0],
    'D-FLOW': np.argsort(evalModels.to_numpy(final_loss_dflow_list))[0],
    'Superposition':10

}

# Colors for each method
colors = {'LGD': 'red', 'LGD-CM': 'green', 'D-FLOW': 'purple',"Superposition":"grey"}

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot theoretical PDF once
x_range = torch.linspace(-10, 10, 1000, device='cuda:0')
pdf_theory = dist_utils.mog_pdf(x_range, mog_means.squeeze(), mog_variances.squeeze(), weights)
ax.plot(x_range.cpu(), pdf_theory.cpu(), 'b-', label='Theoretical', linewidth=5, alpha=0.8)

# Plot each method
for method, best_idx in best_indices.items():
    # Get decoded x_star
    if method == 'LGD':
        decoded_original = best_x_t_LGD_list[best_idx]
    elif method == 'LGD-CM':
        decoded_original = best_x_t_LGD_CM_list[best_idx]
    elif method == 'D-FLOW':
        decoded_original = best_x_t_dflow_list[best_idx]
    else:
        decoded_original = best_x_t_superposition_list[best_idx]
    # Compute conditionals
    mu_cond, Sigma_cond = dist_utils.compute_conditionals(
        mu_list, Sigma_list, decoded_original.reshape(-1, 1)
    )
    alpha_cond = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, decoded_original)

    # Get SWD distance
    swd_norm = norm_swd_dict[method][best_idx]

    # Plot optimized PDF
    pdf_optim = dist_utils.mog_pdf(x_range, mu_cond.squeeze(), Sigma_cond.squeeze(), alpha_cond)
    ax.plot(x_range.cpu(), pdf_optim.detach().cpu().numpy(), '--',
            color=colors[method],
            label=f'{method},  SWD={swd_norm:.4f})',
            linewidth=2)

ax.set_xlabel('Y', fontsize=12)
ax.set_ylabel('PDF', fontsize=12)
ax.set_title('Conditional Distributions: Theoretical vs Optimized Methods', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# OLD: Hard-coded best indices based on final_loss
# CHANGED: Calculate best indices based on 25th percentile of norm_swd
norm_swd_dict = {
    'Superposition': norm_swd_superposition_list,
    'LGD': norm_swd_LGD_list,
    'LGD-CM': norm_swd_LGD_CM_list,
    'D-FLOW': norm_swd_dflow_list
}

best_indices = {}
for method, swd_list in norm_swd_dict.items():
    # CHANGED: Find index closest to 25th percentile
    percentile_25 = np.percentile(swd_list, 1)
    best_idx = np.argmin(np.abs(np.array(swd_list) - percentile_25))
    best_indices[method] = best_idx
    print(best_indices)

# Colors for each method
colors = {'LGD': 'red', 'LGD-CM': 'green', 'D-FLOW': 'purple', "Superposition": "grey"}

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot theoretical PDF once
x_range = torch.linspace(-10, 10, 1000, device='cuda:0')
pdf_theory = dist_utils.mog_pdf(x_range, mog_means.squeeze(), mog_variances.squeeze(), weights)
ax.plot(x_range.cpu(), pdf_theory.cpu(), 'b-', label='Theoretical', linewidth=5, alpha=0.8)

# Plot each method
for method, best_idx in best_indices.items():
    # Get decoded x_star
    if method == 'LGD':
        decoded_original = best_x_t_LGD_list[best_idx]
    elif method == 'LGD-CM':
        decoded_original = best_x_t_LGD_CM_list[best_idx]
    elif method == 'D-FLOW':
        decoded_original = best_x_t_dflow_list[best_idx]
    else:
        decoded_original = best_x_t_superposition_list[best_idx]

    # Compute conditionals
    mu_cond, Sigma_cond = dist_utils.compute_conditionals(
        mu_list, Sigma_list, decoded_original.reshape(-1, 1)
    )
    alpha_cond = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, decoded_original)

    # Get SWD distance
    swd_norm = norm_swd_dict[method][best_idx]

    # Plot optimized PDF
    pdf_optim = dist_utils.mog_pdf(x_range, mu_cond.squeeze(), Sigma_cond.squeeze(), alpha_cond)
    # OLD: label=f'{method},  SWD={swd_norm:.4f})'
    # CHANGED: Fixed label formatting (removed extra parenthesis)
    ax.plot(x_range.cpu(), pdf_optim.detach().cpu().numpy(), '--',
            color=colors[method],
            label=f'{method}, SWD={swd_norm:.4f}',
            linewidth=2)

ax.set_xlabel('Y', fontsize=12)
ax.set_ylabel('PDF', fontsize=12)
ax.set_title('Conditional Distributions: Theoretical vs Optimized Methods', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Prepare data dictionaries
norm_swd_dict = {
    'Superposition': norm_swd_superposition_list,
    'LGD': norm_swd_LGD_list,
    'LGD-CM': norm_swd_LGD_CM_list,
    'D-FLOW': norm_swd_dflow_list
}

# swd_dict = {
#     'Superposition': swd_superposition_list,
#     'LGD': swd_LGD_list,
#     'LGD-CM': swd_LGD_CM_list,
#     'D-FLOW': swd_dflow_list
# }

l1_dict = {
    'Superposition': l1_distance_superposition_list,
    'LGD': l1_distance_LGD_list,
    'LGD-CM': l1_distance_LGD_CM_list,
    'D-FLOW': l1_distance_dflow_list
}

times_dict = {
    'Superposition': [0] * len(norm_swd_superposition_list),
    'LGD': lgd_times,
    'LGD-CM': lgd_cm_times,
    'D-FLOW': dflow_times
}

In [ ]:
def create_summary_table(indices_dict,
                         norm_swd_dict,
                         l1_dict,
                         times_dict,
                         title="Metrics Summary"):
    """
    Create summary statistics table for given indices.

    Parameters:
    -----------
    indices_dict : dict
        {'method_name': [list of indices]}
    norm_swd_dict : dict
        {'method_name': [normalized SWD values]}
    l1_dict : dict
        {'method_name': [L1 distance values]}
    times_dict : dict
        {'method_name': [time values]}
    title : str
        Title for the table
    """
    methods = []
    swd_norm_vals = []
    l1_vals = []
    time_vals = []

    for method, indices in indices_dict.items():
        n = len(indices)
        methods.extend([method] * n)
        # OLD: swd_norm_vals.extend([norm_swd_dict[method][i] for i in indices])
        # CHANGED: Convert to float immediately to ensure numeric type
        swd_norm_vals.extend([float(norm_swd_dict[method][i]) for i in indices])
        l1_vals.extend([float(l1_dict[method][i]) for i in indices])
        time_vals.extend([float(times_dict[method][i]) for i in indices])

    data = {
        'Method': methods,
        'SWD': swd_norm_vals,
        'L1': l1_vals,
        'Time': time_vals
    }

    df = pd.DataFrame(data)

    # OLD: summary = df.groupby('Method').agg({'SWD': ['mean', 'std', 'min', 'max', ('25%', lambda x: np.percentile(x, 25)), ...], ...})
    # CHANGED: Use named aggregation functions without mixing built-ins and lambdas
    summary = df.groupby('Method').agg({
        'SWD': [
            ('mean', 'mean'),
            ('std', 'std'),
            ('min', 'min'),
            ('max', 'max'),
            ('25%', lambda x: np.percentile(x, 25)),
            ('50%', lambda x: np.percentile(x, 50)),
            ('75%', lambda x: np.percentile(x, 75))
        ],
        'L1': [
            ('mean', 'mean'),
            ('std', 'std'),
            ('min', 'min'),
            ('max', 'max'),
            ('25%', lambda x: np.percentile(x, 25)),
            ('50%', lambda x: np.percentile(x, 50)),
            ('75%', lambda x: np.percentile(x, 75))
        ],
        'Time': [
            ('mean', 'mean'),
            ('std', 'std'),
            ('min', 'min'),
            ('max', 'max'),
            ('25%', lambda x: np.percentile(x, 25)),
            ('50%', lambda x: np.percentile(x, 50)),
            ('75%', lambda x: np.percentile(x, 75))
        ]
    }).round(4)

    # Flatten multi-level column names
    summary.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col
                       for col in summary.columns.values]

    print("=" * 70)
    print(title)
    print("=" * 70)
    print(summary)
    print()

    # Time statistics with percentiles
    print("=" * 70)
    print(f"{title} - Time Statistics (seconds)")
    print("=" * 70)
    time_stats = df.groupby('Method')['Time'].agg([
        ('mean', 'mean'),
        ('std', 'std'),
        ('min', 'min'),
        ('max', 'max'),
        ('25%', lambda x: np.percentile(x, 25)),
        ('50%', lambda x: np.percentile(x, 50)),
        ('75%', lambda x: np.percentile(x, 75))
    ]).round(2)
    print(time_stats)
    print("\n")

    return df, summary



# TOP 10 indices
top10_indices = {
    'Superposition': np.random.choice(len(norm_swd_superposition_list), 10, replace=False),
    'LGD': np.argsort(evalModels.to_numpy(final_loss_LGD))[:10],
    'LGD-CM': np.argsort(evalModels.to_numpy(final_loss_LGD_CM))[:10],
    'D-FLOW': np.argsort(evalModels.to_numpy(final_loss_dflow_list))[:10]
}

# ALL indices
all_indices = {
    'Superposition': list(range(len(norm_swd_superposition_list))),
    'LGD': list(range(len(norm_swd_LGD_list))),
    'LGD-CM': list(range(len(norm_swd_LGD_CM_list))),
    'D-FLOW': list(range(len(norm_swd_dflow_list)))
}

# Generate tables
df_top10, summary_top10 = create_summary_table(
    top10_indices, norm_swd_dict,
    l1_dict, times_dict,
    title="TOP 10 - Metrics Summary"
)

df_all, summary_all = create_summary_table(
    all_indices, norm_swd_dict,
    l1_dict, times_dict,
    title="ALL ATTEMPTS - Metrics Summary"
)



In [ ]:
df_top10

In [ ]:
summary_top10

In [ ]:
df_all

In [ ]:
summary_all

In [ ]:
# Usage example:
models_config = {
    'Diffusion_Conditional': {
        'model': model_cond,
        'type': 'diffusion',
        'task': 'P(Y|X)',
        'extract_generated_only': True,
        'reverse_distribution': False
    },
    'FlowMatching_Conditional': {
        'model': vf_y_cond_x,
        'type': 'fm',
        'task': 'P(Y|X)',
        'extract_generated_only': False,
        'reverse_distribution': False
    },
    'ConsistencyModel': {
        'model': Cos_ConsistencyModeliCT,
        'type': 'cm',
        'task': 'P(Y|X)',
        'extract_generated_only': False,
        'reverse_distribution': False
    },
    'SuperDiffusion': {
        'model': model_cond_diff,
        'type': 'super_diffusion',
        'task': 'P(X|Y)',
        'extract_generated_only': True,
        'reverse_distribution': True
    },
    'Diffusion_Unconditional': {
        'model': model_uncond,
        'type': 'diffusion',
        'task': 'P(X)',
        'extract_generated_only': False,
        'reverse_distribution': False
    },
    'FlowMatching_Unconditional': {
        'model': vf_X,
        'type': 'fm',
        'task': 'P(X)',
        'extract_generated_only': False,
        'reverse_distribution': False
    },
}


# Run evaluation
df_results, summary_stats = evalModels.create_model_evaluation_summary(
    models_dict=models_config,
    mu_list=mu_list,
    Sigma_list=Sigma_list,
    alpha=alpha,
    condition_on=condition_on,
    device=device,
    num_evaluations=250,
    nsamples=2_500,
    num_projections=1000
)

In [ ]:
# Usage example:
models_config = {

    'FlowMatching_Conditional': {
        'model': vf_y_cond_x,
        'type': 'fm',
        'task': 'P(Y|X)',
        'extract_generated_only': False,
        'reverse_distribution': False
    },
    'ConsistencyModel': {
        'model': Cos_ConsistencyModeliCT,
        'type': 'cm',
        'task': 'P(Y|X)',
        'extract_generated_only': False,
        'reverse_distribution': False
    }
}


# Run evaluation
df_results, summary_stats = evalModels.create_model_evaluation_summary(
    models_dict=models_config,
    mu_list=mu_list,
    Sigma_list=Sigma_list,
    alpha=alpha,
    condition_on=condition_on,
    device=device,
    num_evaluations=250,
    nsamples=2_500,
    num_projections=1000
)

In [ ]:
import matplotlib.pyplot as plt

# change
# Create boxplot for all four distance lists
data = [l1_distance_dflow_list, l1_distance_superposition_list, l1_distance_LGD_list, l1_distance_LGD_CM_list]

labels = ['DFlow', 'Superposition', 'LGD', 'LGD_CM']

plt.figure(figsize=(12, 6))
plt.boxplot(data, labels=labels)
plt.ylabel('L1 Distance')
plt.title('L1 Distance Comparison - All Methods')
plt.grid(True, alpha=0.3)
plt.show()
# changed

In [ ]:
# Plot optimization times
time_data = [dflow_times, superposition_times, lgd_times, lgd_cm_times]
time_labels = ['DFlow', 'Superposition', 'LGD', 'LGD_CM']

plt.figure(figsize=(12, 6))
plt.boxplot(time_data, labels=time_labels)
plt.ylabel('Optimization Time (seconds)')
plt.title('Optimization Time Comparison')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
from matplotlib.ticker import LogLocator, LogFormatter, FixedLocator

# EMD Distance Comparison with dynamic range and log scale
emd_data = [norm_swd_dflow_list, norm_swd_superposition_list,
          norm_swd_LGD_list, norm_swd_LGD_CM_list]
method_names = ['D-Flow', 'Superposition', 'LGD', 'LGD-CM']

# Find max value across all EMD data for dynamic range
all_emd_values = []
for distances in emd_data:
   flat = np.concatenate([np.array(d).flatten() for d in distances])
   all_emd_values.extend(flat)
max_val = max(all_emd_values)
bins = np.arange(0, max_val + 0.05, 0.05)


# EMD Histograms with log scale and more ticks
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, (distances, name) in enumerate(zip(emd_data, method_names)):
   flat = np.concatenate([np.array(d).flatten() for d in distances])
   axes.flat[i].hist(flat, bins=bins, density=True, alpha=0.7, edgecolor='black')
   axes.flat[i].set_title(name)
   axes.flat[i].set_xscale('log')  # Log scale for x-axis

   # OLD: axes.flat[i].xaxis.set_major_locator(LogLocator(base=10, numticks=10))
   # CHANGED: Use FixedLocator for custom tick positions
   custom_ticks = [0.1, 0.25, 0.5, 0.75, 1, 2, 5, 10, 20, 50, 100]
   axes.flat[i].xaxis.set_major_locator(FixedLocator(custom_ticks))
   axes.flat[i].xaxis.set_minor_locator(LogLocator(base=10, subs=[2, 3, 4, 5, 6, 7, 8, 9], numticks=50))
   # OLD: axes.flat[i].xaxis.set_major_formatter(LogFormatter(labelOnlyBase=False))
   # CHANGED: Use ScalarFormatter to show actual numbers
   from matplotlib.ticker import ScalarFormatter
   axes.flat[i].xaxis.set_major_formatter(ScalarFormatter())
   axes.flat[i].xaxis.set_minor_formatter(LogFormatter(labelOnlyBase=False, minor_thresholds=(2, 0.4)))

   # Rotate tick labels for better readability
   axes.flat[i].tick_params(axis='x', rotation=45, which='both')
fig.suptitle('EMD Comparison (Log Scale)', fontsize=16)
plt.tight_layout()
plt.show()

# EMD ECDF with log scale on x-axis
plt.figure(figsize=(10, 6))
colors = ['blue', 'orange', 'green', 'red']
for distances, name, color in zip(emd_data, method_names, colors):
   flat = np.concatenate([np.array(d).flatten() for d in distances])
   sorted_data = np.sort(flat)
   ecdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
   plt.plot(sorted_data, ecdf, label=name, color=color, linewidth=2.5)
plt.xlabel('EMD Distance (log scale)')
plt.ylabel('Cumulative Probability')
plt.title('ECDF - EMD Distance Comparison')
plt.xscale('log')
# OLD: plt.gca().xaxis.set_major_locator(LogLocator(base=10, numticks=10))
# CHANGED: Use FixedLocator for custom tick positions
custom_ticks = [0.1, 0.25, 0.5, 0.75, 1, 2, 5, 10, 20, 50, 100]
plt.gca().xaxis.set_major_locator(FixedLocator(custom_ticks))
plt.gca().xaxis.set_minor_locator(LogLocator(base=10, subs=[2, 3, 4, 5, 6, 7, 8, 9], numticks=50))
# OLD: plt.gca().xaxis.set_major_formatter(LogFormatter(labelOnlyBase=False))
# CHANGED: Use ScalarFormatter to show actual numbers
from matplotlib.ticker import ScalarFormatter
plt.gca().xaxis.set_major_formatter(ScalarFormatter())
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## SuperPosition Compare different number of conditions

### Train

In [ ]:
# Flip the values for superposition
point_l = [p.flip(0) for p in mu_list]
mu_list_super = point_l
idx = torch.arange(Sigma_list[0].size(0) - 1, -1, -1)
Sigma_list_super = [s[idx][:, idx]  for s in Sigma_list]# [.T for sigma in Sigma_list]


nfeatures= mu_list[0].shape[0]
condition_on=1

# Train
model_cond_diff =DiffusionGeneral.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=True,condition_on=condition_on,diffusion_steps=diffusion_steps)
model_cond_diff.train_model( data_generator=partial(dist_utils.generate_mog_samples, xi=mu_list_super, S=Sigma_list_super, beta=alpha, kernel_func=None),dataloader=None, num_iterations=nepochs)


In [ ]:
mu_list, Sigma_list, alpha,mog_means, mog_variances,weights,x_star= dist_utils.get_param_mog_with_target(dim_data=10,num_components=4,device='cpu',conditional_modes=2,distanceOrScale="Distance")
mog_means, mog_variances, weights= dist_utils.filter_and_normalize(mog_means, mog_variances, weights, threshold=0.001)


In [ ]:
n_sample=n_attemp_optim=500
first_value = mog_means[0].item()
second_value = mog_means[1].item()
cond1= torch.tensor([first_value],device=device)
cond2= torch.tensor([second_value],device=device)

start_time = time.time()
cond_l= [cond1,cond2]
x_gen_iso,_=  superposition_utils.generate_isosurface_samples(model_cond_diff, cond_l=cond_l,bs=n_sample,device=device)
X_gen_np = x_gen_iso[:, -1, :].detach().cpu().numpy()
end_time = time.time()


superposition_times_cond2 = [end_time - start_time] *n_attemp_optim


# Flip again
X_gen_np_original = torch.tensor(X_gen_np)[:,condition_on:]
X_gen_np_original=X_gen_np_original.flip(1)


In [ ]:
X_gen_np_original

In [ ]:
l1_distance_superposition_list_2cond=[]
best_x_t_superposition_list_2cond=[]
for i in range(X_gen_np_original.shape[0]):
   val = X_gen_np_original[i:i+1].reshape(-1, 1)
   l1_distance = dist_utils.warpper_L1_distance(val, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
   l1_distance_superposition_list_2cond.append(l1_distance )
   best_x_t_superposition_list_2cond.append(val)


print(best_x_t_superposition_list_2cond)
print(l1_distance_superposition_list_2cond)

plt.hist(l1_distance_superposition_list_2cond,  edgecolor='black',range=(0,2),bins=100,)
plt.show()


In [ ]:
mog_means

In [ ]:
n_sample=n_attemp_optim=500
first_value = mog_means[0].item()
second_value = mog_means[1].item()
cond1= torch.tensor([first_value],device=device)
cond2= torch.tensor([second_value],device=device)
cond3= torch.tensor([first_value+0.2],device=device)

start_time = time.time()
cond_l= [cond1,cond2,cond3]
x_gen_iso,_=  superposition_utils.generate_isosurface_samples(model_cond_diff, cond_l=cond_l,bs=n_sample,device=device)
X_gen_np = x_gen_iso[:, -1, :].detach().cpu().numpy()
end_time = time.time()


superposition_times_cond3 = [end_time - start_time] *n_attemp_optim


# Flip again
X_gen_np_original = torch.tensor(X_gen_np)[:,condition_on:]
X_gen_np_original=X_gen_np_original.flip(1)


In [ ]:
best_x_t_superposition_list_3cond=[]
l1_distance_superposition_list_3cond=[]
for i in range(X_gen_np_original.shape[0]):
   val = X_gen_np_original[i:i+1].reshape(-1, 1)
   l1_distance = dist_utils.warpper_L1_distance(val, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
   l1_distance_superposition_list_3cond.append(l1_distance )
   best_x_t_superposition_list_3cond.append(val)


print(best_x_t_superposition_list_3cond)
print(l1_distance_superposition_list_3cond)

plt.hist(l1_distance_superposition_list_3cond,  edgecolor='black',range=(0,2),bins=100,)
plt.show()


In [ ]:
n_sample=n_attemp_optim=500
first_value = mog_means[0].item()
second_value = mog_means[1].item()
cond1= torch.tensor([first_value],device=device)
cond2= torch.tensor([second_value],device=device)
cond3= torch.tensor([first_value+0.2],device=device)
cond4= torch.tensor([second_value-0.1],device=device)

start_time = time.time()
cond_l= [cond1,cond2,cond3,cond4]
x_gen_iso,_=  superposition_utils.generate_isosurface_samples(model_cond_diff, cond_l=cond_l,bs=n_sample,device=device)
X_gen_np = x_gen_iso[:, -1, :].detach().cpu().numpy()
end_time = time.time()


superposition_times_cond4 = [end_time - start_time] *n_attemp_optim


# Flip again
X_gen_np_original = torch.tensor(X_gen_np)[:,condition_on:]
X_gen_np_original=X_gen_np_original.flip(1)


In [ ]:
best_x_t_superposition_list_4cond=[]
l1_distance_superposition_list_4cond=[]
for i in range(X_gen_np_original.shape[0]):
   val = X_gen_np_original[i:i+1].reshape(-1, 1)
   l1_distance = dist_utils.warpper_L1_distance(val, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
   l1_distance_superposition_list_4cond.append(l1_distance )
   best_x_t_superposition_list_4cond.append(val)


print(best_x_t_superposition_list_4cond)
print(l1_distance_superposition_list_4cond)

plt.hist(l1_distance_superposition_list_4cond,  edgecolor='black',range=(0,2),bins=100,)
plt.show()


In [ ]:
n_sample=n_attemp_optim=500
first_value = mog_means[0].item()
second_value = mog_means[1].item()
cond1= torch.tensor([first_value],device=device)
cond2= torch.tensor([second_value],device=device)
cond3= torch.tensor([first_value+0.2],device=device)
cond4= torch.tensor([second_value-0.1],device=device)
cond5= torch.tensor([second_value+0.3],device=device)

start_time = time.time()
cond_l= [cond1,cond2,cond3,cond4,cond5]
x_gen_iso,_=  superposition_utils.generate_isosurface_samples(model_cond_diff, cond_l=cond_l,bs=n_sample,device=device)
X_gen_np = x_gen_iso[:, -1, :].detach().cpu().numpy()
end_time = time.time()


superposition_times_cond5 = [end_time - start_time] *n_attemp_optim


# Flip again
X_gen_np_original = torch.tensor(X_gen_np)[:,condition_on:]
X_gen_np_original=X_gen_np_original.flip(1)


In [ ]:
best_x_t_superposition_list_5cond=[]
l1_distance_superposition_list_5cond=[]
for i in range(X_gen_np_original.shape[0]):
   val = X_gen_np_original[i:i+1].reshape(-1, 1)
   l1_distance = dist_utils.warpper_L1_distance(val, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
   l1_distance_superposition_list_5cond.append(l1_distance )
   best_x_t_superposition_list_5cond.append(val)


print(best_x_t_superposition_list_5cond)
print(l1_distance_superposition_list_5cond)

plt.hist(l1_distance_superposition_list_5cond,  edgecolor='black',range=(0,2),bins=100,)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Updated lists with Condition 5
lists = [
    l1_distance_superposition_list_5cond,
    l1_distance_superposition_list_4cond,
    l1_distance_superposition_list_3cond,
    l1_distance_superposition_list_2cond
]

labels = ["5 Conditions", "4 Conditions", "3 Conditions", "2 Conditions"]
colors = ['purple', 'blue', 'green', 'red']

# Filter each list to include values between 0 and 2
arrays = [
    np.array([float(x) for x in lst if 0 <= float(x) <= 2], dtype=np.float32)
    for lst in lists
]

# Compute stats
means = [np.mean(arr) for arr in arrays]
medians = [np.median(arr) for arr in arrays]
stds = [np.std(arr) for arr in arrays]

# Create and print the table
df_l1 = pd.DataFrame({
    "Condition": labels,
    "Mean": means,
    "Median": medians,
    "Std": stds
})
print("L1 Distance Statistics (Filtered 0 ≤ x ≤ 2):")

# Plot ECDF
plt.figure(figsize=(10, 6))
for arr, label, color in zip(arrays, labels, colors):
    sorted_arr = np.sort(arr)
    ecdf = np.arange(1, len(sorted_arr) + 1) / len(sorted_arr)
    plt.plot(sorted_arr, ecdf, marker='.', linestyle='-', label=label, color=color)

plt.xlabel('L1 Distance')
plt.ylabel('ECDF')
plt.title('ECDF of L1 Distances  - Compare Differnert Conditions in Superposition - 500 samples')
plt.grid(True, alpha=0.3)
plt.xlim(0, 2)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Time data (no filtering)
time_lists = [
    superposition_times_cond5,
    superposition_times_cond4,
    superposition_times_cond3,
    superposition_times_cond2
]

time_labels = ["5 Conditions", "4 Conditions", "3 Conditions", "2 Conditions"]

# Convert to NumPy arrays
arrays = [np.array([float(x) for x in lst], dtype=np.float32) for lst in time_lists]

# Compute stats
means = [np.mean(arr) for arr in arrays]
medians = [np.median(arr) for arr in arrays]
stds = [np.std(arr) for arr in arrays]

# Create and print table
df_time = pd.DataFrame({
    "Condition": time_labels,
    "Mean (s)": means,
    "Median (s)": medians,
    "Std (s)": stds
})
df_time

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

# Step 1: Convert l1 distances to regular numbers
l1_distances = [l.detach().cpu().item() for l in l1_distance_superposition_list_5cond]

# Step 2: Define target values and find closest indices
targets = [0.2, 0.4, 0.6, 0.8, 1., 1.25, 1.5, 2]
closest_indices = [np.argmin([abs(d - t) for d in l1_distances]) for t in targets]

# Step 3: Prepare x-axis values for PDF plots
x_vals = torch.linspace(-20, 10, 1000).unsqueeze(1)  # shape [1000, 1]

# Step 4: Create subplots
fig, axs = plt.subplots(2, 4, figsize=(20, 8))
axs = axs.flatten()

all_x = []
all_y = []

# Step 5: Iterate through each index, compute PDFs, and store values
for i, idx in enumerate(closest_indices):
    best_sample = best_x_t_superposition_list_5cond[idx]

    # Compute conditional distribution
    mu_temp, Sigma_temp = dist_utils.compute_conditionals(mu_list, Sigma_list, best_sample)
    temp_alpha = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, best_sample)

    means_best = [m.view(-1) for m in mu_temp]
    covariances_best = [c.view(-1) for c in Sigma_temp]
    temp_alpha_best = temp_alpha.squeeze()

    # Compute PDFs
    pdf_val_good = dist_utils.mog_multivariate_pdf(x_vals, means_best, covariances_best, temp_alpha_best, log=False)
    pdf_vals_requested = dist_utils.mog_multivariate_pdf(x_vals, mog_means, mog_variances, weights, log=False)

    # Move to CPU for plotting
    pdf_val_good = pdf_val_good.detach().cpu().numpy()
    pdf_vals_requested = pdf_vals_requested.detach().cpu().numpy()
    x_vals_np = x_vals.squeeze(1).cpu().numpy()

    # Store for global min/max calculation
    all_x.append(x_vals_np)
    all_y.append(pdf_val_good)
    all_y.append(pdf_vals_requested)

    # Plot
    axs[i].plot(x_vals_np, pdf_val_good, label=f"PDF - l1={targets[i]:.2f}")
    axs[i].plot(x_vals_np, pdf_vals_requested, label="Requested PDF")
    axs[i].legend()
    axs[i].set_title(f"L1={targets[i]}")
    axs[i].set_xlabel("x")
    axs[i].set_ylabel("PDF")

# Step 6: Set same x/y limits for all subplots
x_min, x_max = np.min(all_x), np.max(all_x)
y_min, y_max = np.min(all_y), np.max(all_y)

for ax in axs:
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

plt.suptitle("Comparison of Mixture PDFs Across Different L1 Distances", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import torch

# ============================================
# 2D COND 1D - OPTIMAL X = -5
# ============================================
optimal_2d = -5.0

# CHANGED: Use torch.tensor instead of tensor
# OLD: superposition_2d_xt = [tensor([-4.8964]), tensor([-4.8967]), ...]
# SUPERPOSITION
superposition_2d_xt = [torch.tensor([-4.8964]), torch.tensor([-4.8967]), torch.tensor([-4.9090]), torch.tensor([-4.8884]),
                       torch.tensor([-4.8710]), torch.tensor([-4.8775]), torch.tensor([-4.9099]), torch.tensor([-4.8895]),
                       torch.tensor([-4.8633]), torch.tensor([-4.8688]), torch.tensor([-4.8911]), torch.tensor([-4.9023]),
                       torch.tensor([-4.8688]), torch.tensor([-4.8939]), torch.tensor([-4.9170]), torch.tensor([-4.8820]),
                       torch.tensor([-4.8962]), torch.tensor([-4.8994]), torch.tensor([-4.8820]), torch.tensor([-4.8905]),
                       torch.tensor([-4.9192]), torch.tensor([-4.8898]), torch.tensor([-4.9287]), torch.tensor([-4.8848]),
                       torch.tensor([-4.9057])]
# END CHANGED

superposition_2d_swd = [np.float32(0.028294338), np.float32(0.009951461), np.float32(0.036783747),
                        np.float32(0.009733002), np.float32(0.035735957), np.float32(0.023430815),
                        np.float32(0.01772575), np.float32(0.013965154), np.float32(0.013393749),
                        np.float32(0.019052522), np.float32(0.02046474), np.float32(0.030391134),
                        np.float32(0.0154624125), np.float32(0.024910968), np.float32(0.0123252645),
                        np.float32(0.039296597), np.float32(0.009898568), np.float32(0.031973857),
                        np.float32(0.015989788), np.float32(0.015693948), np.float32(0.010325503),
                        np.float32(0.011645749), np.float32(0.013605441), np.float32(0.026436748),
                        np.float32(0.01301936)]

# CHANGED: Use torch.tensor instead of tensor
# OLD: lgdcm_2d_xt = [tensor([[-4.5123]], device='cuda:0'), ...]
# LGD-CM
lgdcm_2d_xt = [torch.tensor([[-4.5123]], device='cuda:0'), torch.tensor([[254.3475]], device='cuda:0'),
               torch.tensor([[-4.4408]], device='cuda:0'), torch.tensor([[-4.3564]], device='cuda:0'),
               torch.tensor([[-5.3480]], device='cuda:0'), torch.tensor([[3.7036]], device='cuda:0'),
               torch.tensor([[-4.3607]], device='cuda:0'), torch.tensor([[-4.2860]], device='cuda:0'),
               torch.tensor([[-26.9201]], device='cuda:0'), torch.tensor([[-4.4434]], device='cuda:0'),
               torch.tensor([[-4.4043]], device='cuda:0'), torch.tensor([[-4.0843]], device='cuda:0'),
               torch.tensor([[-5.4444]], device='cuda:0'), torch.tensor([[-32.0487]], device='cuda:0'),
               torch.tensor([[-4.2097]], device='cuda:0'), torch.tensor([[-4.9054]], device='cuda:0'),
               torch.tensor([[-285.1108]], device='cuda:0'), torch.tensor([[-4.2945]], device='cuda:0'),
               torch.tensor([[-4.3885]], device='cuda:0'), torch.tensor([[21.5294]], device='cuda:0'),
               torch.tensor([[-4.3794]], device='cuda:0'), torch.tensor([[-4.2163]], device='cuda:0'),
               torch.tensor([[-4.2887]], device='cuda:0'), torch.tensor([[-4.8632]], device='cuda:0'),
               torch.tensor([[-4.3917]], device='cuda:0')]
# END CHANGED

lgdcm_2d_swd = [np.float32(0.056510996), np.float32(19.871117), np.float32(0.05199985),
                np.float32(0.052113727), np.float32(0.056380194), np.float32(0.6006926),
                np.float32(0.054631207), np.float32(0.06384025), np.float32(1.357697),
                np.float32(0.057100497), np.float32(0.045626048), np.float32(0.08262813),
                np.float32(0.076684594), np.float32(1.7346872), np.float32(0.06997574),
                np.float32(0.009883697), np.float32(22.104244), np.float32(0.06139482),
                np.float32(0.052917417), np.float32(2.1344025), np.float32(0.06130995),
                np.float32(0.06954629), np.float32(0.057783104), np.float32(0.021526504),
                np.float32(0.0687377)]

# CHANGED: Use torch.tensor instead of tensor
# OLD: dflow_2d_xt_all = [tensor([[-4.6943]], device='cuda:0'), ...]
# D-FLOW (every other one is duplicate, so take every 2nd)
dflow_2d_xt_all = [torch.tensor([[-4.6943]], device='cuda:0'), torch.tensor([[-5.1526]], device='cuda:0'),
                   torch.tensor([[-4.7446]], device='cuda:0'), torch.tensor([[-4.3974]], device='cuda:0'),
                   torch.tensor([[-4.8506]], device='cuda:0'), torch.tensor([[-2.1662]], device='cuda:0'),
                   torch.tensor([[-4.3702]], device='cuda:0'), torch.tensor([[-4.5939]], device='cuda:0'),
                   torch.tensor([[0.8542]], device='cuda:0'), torch.tensor([[6.2200]], device='cuda:0'),
                   torch.tensor([[-5.2046]], device='cuda:0'), torch.tensor([[-5.4788]], device='cuda:0'),
                   torch.tensor([[-5.1864]], device='cuda:0'), torch.tensor([[-4.6755]], device='cuda:0'),
                   torch.tensor([[-5.1824]], device='cuda:0'), torch.tensor([[-4.5975]], device='cuda:0'),
                   torch.tensor([[0.4512]], device='cuda:0'), torch.tensor([[-4.6889]], device='cuda:0'),
                   torch.tensor([[0.5884]], device='cuda:0'), torch.tensor([[4.7296]], device='cuda:0'),
                   torch.tensor([[-1.2278]], device='cuda:0'), torch.tensor([[-4.1912]], device='cuda:0'),
                   torch.tensor([[-4.8585]], device='cuda:0'), torch.tensor([[-4.8478]], device='cuda:0'),
                   torch.tensor([[6.2032]], device='cuda:0')]
# END CHANGED

dflow_2d_swd_all = [np.float32(0.035653938), np.float32(0.027245369), np.float32(0.020437561),
                    np.float32(0.05027806), np.float32(0.033591382), np.float32(0.30362758),
                    np.float32(0.057367142), np.float32(0.03458123), np.float32(0.44993436),
                    np.float32(0.6902522), np.float32(0.037160978), np.float32(0.08353233),
                    np.float32(0.032789756), np.float32(0.028254407), np.float32(0.028675757),
                    np.float32(0.032679535), np.float32(0.544058), np.float32(0.031039461),
                    np.float32(0.5099011), np.float32(0.6030967), np.float32(0.37454012),
                    np.float32(0.09009274), np.float32(0.021805), np.float32(0.015122181),
                    np.float32(0.68897957)]

# CHANGED: Use torch.tensor instead of tensor
# OLD: lgd_2d_xt = [tensor([[-5.0518]], device='cuda:0'), ...]
# LGD
lgd_2d_xt = [torch.tensor([[-5.0518]], device='cuda:0'), torch.tensor([[-4.8967]], device='cuda:0'),
             torch.tensor([[-4.9385]], device='cuda:0'), torch.tensor([[5.8375]], device='cuda:0'),
             torch.tensor([[6.1446]], device='cuda:0'), torch.tensor([[-4.2642]], device='cuda:0'),
             torch.tensor([[-55.4033]], device='cuda:0'), torch.tensor([[-4.9040]], device='cuda:0'),
             torch.tensor([[-4.9725]], device='cuda:0'), torch.tensor([[-4.3212]], device='cuda:0'),
             torch.tensor([[-4.8021]], device='cuda:0'), torch.tensor([[-4.5568]], device='cuda:0'),
             torch.tensor([[-4.7816]], device='cuda:0'), torch.tensor([[-4.8945]], device='cuda:0'),
             torch.tensor([[-4.7680]], device='cuda:0'), torch.tensor([[-4.8854]], device='cuda:0'),
             torch.tensor([[-4.9585]], device='cuda:0'), torch.tensor([[-4.9949]], device='cuda:0'),
             torch.tensor([[-4.6957]], device='cuda:0'), torch.tensor([[-4.3867]], device='cuda:0'),
             torch.tensor([[-4.9347]], device='cuda:0'), torch.tensor([[-4.8666]], device='cuda:0'),
             torch.tensor([[-4.8938]], device='cuda:0'), torch.tensor([[-4.9644]], device='cuda:0'),
             torch.tensor([[-4.9767]], device='cuda:0')]
# END CHANGED

lgd_2d_swd = [np.float32(0.016842952), np.float32(0.010149586), np.float32(0.021128185),
              np.float32(0.61219954), np.float32(0.66307753), np.float32(0.057665706),
              np.float32(4.25352), np.float32(0.020010265), np.float32(0.006906169),
              np.float32(0.054067556), np.float32(0.01856816), np.float32(0.03509322),
              np.float32(0.028905673), np.float32(0.012861935), np.float32(0.027037647),
              np.float32(0.0148300305), np.float32(0.011130163), np.float32(0.009587374),
              np.float32(0.031453043), np.float32(0.05604909), np.float32(0.012058636),
              np.float32(0.011920721), np.float32(0.018038467), np.float32(0.019205919),
              np.float32(0.00984569)]

# ============================================
# CHANGED: Extract values from tensors
# ============================================
def extract_value(tensor_val):
    """Extract scalar value from tensor"""
    if isinstance(tensor_val, torch.Tensor):
        return tensor_val.cpu().detach().item()
    return float(tensor_val)

def calculate_l2_distances_2d(xt_list, swd_list, optimal, name):
    """Calculate L2 distances for 2D case"""
    # CHANGED: Extract values from tensors
    # OLD: xt_values = [x.cpu().detach().item() if isinstance(x, torch.Tensor) else x for x in xt_list]
    xt_values = [extract_value(x) for x in xt_list]
    # END CHANGED

    # Calculate L2 distances
    l2_distances = [abs(x - optimal) for x in xt_values]

    # Get top 10 indices based on SWD
    top10_indices = np.argsort(swd_list)[:10]

    # All attempts
    all_mean = np.mean(l2_distances)
    all_std = np.std(l2_distances)

    # Top 10 attempts
    top10_l2 = [l2_distances[i] for i in top10_indices]
    top10_mean = np.mean(top10_l2)
    top10_std = np.std(top10_l2)

    print(f"\n{name}:")
    print(f"  All attempts: Mean L2 = {all_mean:.4f}, Std L2 = {all_std:.4f}")
    print(f"  Top 10 SWD:   Mean L2 = {top10_mean:.4f}, Std L2 = {top10_std:.4f}")

    return {
        'all_mean': all_mean,
        'all_std': all_std,
        'top10_mean': top10_mean,
        'top10_std': top10_std
    }

print("="*60)
print("2D CONDITIONAL 1D - L2 DISTANCES TO OPTIMAL (x = -5)")
print("="*60)

results_2d = {}
results_2d['SUPERPOSITION'] = calculate_l2_distances_2d(superposition_2d_xt, superposition_2d_swd, optimal_2d, "SUPERPOSITION")
results_2d['LGD-CM'] = calculate_l2_distances_2d(lgdcm_2d_xt, lgdcm_2d_swd, optimal_2d, "LGD-CM")
results_2d['D-FLOW'] = calculate_l2_distances_2d(dflow_2d_xt_all, dflow_2d_swd_all, optimal_2d, "D-FLOW")
results_2d['LGD'] = calculate_l2_distances_2d(lgd_2d_xt, lgd_2d_swd, optimal_2d, "LGD")

In [ ]:
import numpy as np
import torch

# ============================================
# 10D COND 9D - OPTIMAL X
# ============================================
optimal_10d = torch.tensor([-4.5404, 2.7114, 0.5513, -11.2950, 3.0335, -0.6915, 4.1551, -1.2385, -4.0147])

# CHANGED: Added complete 10D data with torch.tensor
# OLD: Data not included
# SUPERPOSITION
superposition_10d_xt = [
    torch.tensor([[-5.0088], [-1.2863], [0.5484], [-8.2673], [1.2691], [0.2184], [1.3511], [-1.9405], [-4.8379]]),
    torch.tensor([[-2.4817], [0.7172], [-2.6161], [-11.6680], [0.8050], [-0.4443], [2.1969], [1.7631], [-7.4905]]),
    torch.tensor([[-3.1857], [3.7396], [-3.1043], [-13.6954], [1.6284], [-3.2570], [-9.3741], [-0.1939], [-1.8457]]),
    torch.tensor([[-9.4334], [0.3152], [0.2952], [-9.8948], [-1.5460], [-1.8473], [5.2181], [2.9442], [-6.1876]]),
    torch.tensor([[-4.4834], [1.3526], [1.6760], [-7.0602], [2.6923], [0.9501], [0.3844], [-1.4195], [-7.4886]]),
    torch.tensor([[-2.9094], [2.2146], [-0.7923], [-10.1675], [2.6275], [1.0471], [1.4372], [0.9337], [-8.3997]]),
    torch.tensor([[-5.7885], [-0.1087], [1.9946], [-9.5734], [3.2950], [0.5646], [-2.4307], [-8.3054], [-6.9282]]),
    torch.tensor([[-4.5974], [1.6625], [-0.0428], [-8.6150], [2.1716], [0.8041], [0.7214], [-1.0316], [-7.6527]]),
    torch.tensor([[-0.2952], [3.6204], [-2.9292], [-9.9627], [2.8203], [0.5124], [0.0475], [-0.4967], [-6.8670]]),
    torch.tensor([[-3.4100], [4.4103], [-1.0515], [-12.9600], [3.2405], [2.0638], [2.8505], [-0.1766], [-2.6719]]),
    torch.tensor([[-2.3277], [4.1353], [1.5718], [-11.0706], [2.3122], [0.0308], [2.1346], [-0.7466], [-5.2093]]),
    torch.tensor([[-4.7427], [-0.2162], [1.2076], [-5.6920], [1.9422], [0.9770], [-1.9091], [-0.7179], [-5.1345]]),
    torch.tensor([[-1.9405], [4.0178], [-1.3819], [-11.9193], [2.2457], [-1.3697], [-5.7689], [0.8960], [-2.8429]]),
    torch.tensor([[-3.9561], [3.6694], [2.3793], [-9.6075], [2.3849], [-2.7000], [2.0688], [-0.5553], [-6.1956]]),
    torch.tensor([[-0.7320], [2.7133], [-6.2857], [-14.7348], [1.9577], [-0.8733], [-2.6574], [6.2304], [-4.9072]]),
    torch.tensor([[-3.5608], [0.5780], [0.7989], [-7.6590], [1.3529], [0.5601], [3.2141], [1.3339], [-11.3812]]),
    torch.tensor([[0.7928], [0.7772], [-7.4383], [-13.6715], [2.4198], [-3.1678], [-6.7437], [5.1954], [-1.8610]]),
    torch.tensor([[-4.9647e+00], [3.1155e+00], [9.4929e-01], [-9.8953e+00], [1.7687e+00], [-2.8988e+00], [1.9005e-01], [9.7921e-03], [-4.5663e+00]]),
    torch.tensor([[-4.8384], [1.8110], [0.0581], [-10.9753], [1.4277], [-0.9015], [2.3786], [-1.0546], [-3.6442]]),
    torch.tensor([[-3.4717], [0.0126], [0.3332], [-7.4856], [2.6143], [2.5859], [-1.0009], [-0.0837], [-5.7519]]),
    torch.tensor([[-3.0879], [-3.5253], [-1.4059], [-10.1958], [1.8100], [2.7211], [-0.1565], [-5.4942], [-3.0445]]),
    torch.tensor([[-3.8989], [3.8519], [0.2835], [-12.1904], [3.6714], [-0.4276], [4.1720], [-0.7149], [-3.0061]]),
    torch.tensor([[-3.8390], [2.7702], [2.2348], [-9.0998], [3.7193], [1.0090], [3.3772], [-1.2828], [-6.4495]]),
    torch.tensor([[-2.2840], [4.7905], [1.8174], [-10.7655], [5.6799], [-0.3013], [1.0878], [-2.1062], [-2.5277]]),
    torch.tensor([[-5.9925], [-0.5883], [0.1192], [-5.8319], [1.3260], [0.7704], [-1.9214], [1.1384], [-5.7387]])
]

superposition_10d_swd = [np.float32(0.8493598), np.float32(0.76250637), np.float32(3.3056018),
                         np.float32(0.63076586), np.float32(0.42627186), np.float32(1.5219105),
                         np.float32(1.4001929), np.float32(0.6649485), np.float32(1.338581),
                         np.float32(0.06812125), np.float32(0.36296448), np.float32(1.1251796),
                         np.float32(1.8556228), np.float32(0.388339), np.float32(2.3525941),
                         np.float32(1.230691), np.float32(5.0231547), np.float32(0.60398084),
                         np.float32(0.10493783), np.float32(1.189135), np.float32(0.7066245),
                         np.float32(0.11856685), np.float32(0.35591426), np.float32(0.36793748),
                         np.float32(1.2153754)]

# LGD
lgd_10d_xt = [
    torch.tensor([[-4.4835], [3.2063], [-0.2412], [-10.0068], [5.7882], [2.4297], [3.1730], [-2.5715], [-4.6328]], device='cuda:0'),
    torch.tensor([[-4.2479], [2.9333], [-1.5878], [-11.5246], [4.7729], [2.8783], [3.4926], [-1.9042], [-2.9615]], device='cuda:0'),
    torch.tensor([[-4.5319], [3.2787], [-1.9741], [-12.5772], [3.8336], [2.9259], [4.9727], [-0.9216], [-3.8454]], device='cuda:0'),
    torch.tensor([[-4.6518], [2.9328], [-1.7279], [-11.2576], [4.1110], [3.1647], [3.1497], [-1.3110], [-4.0481]], device='cuda:0'),
    torch.tensor([[-3.8667], [2.7540], [-0.4127], [-10.4970], [4.9740], [2.5383], [1.3839], [-2.6015], [-2.6562]], device='cuda:0'),
    torch.tensor([[-4.3909], [3.3240], [-1.8405], [-11.7225], [4.7733], [2.7478], [3.5038], [-1.7536], [-3.0861]], device='cuda:0'),
    torch.tensor([[-4.7315], [5.9179], [-1.9892], [-12.6525], [3.0050], [6.1557], [1.2813], [-4.5754], [-5.1244]], device='cuda:0'),
    torch.tensor([[0.5012], [-7.6836], [-9.9058], [1.2616], [6.9711], [1.5267], [-0.2901], [-10.2936], [-10.0116]], device='cuda:0'),
    torch.tensor([[-4.6001], [3.2670], [-1.8699], [-11.6728], [4.2436], [3.0500], [4.0731], [-1.2512], [-3.8897]], device='cuda:0'),
    torch.tensor([[-4.4499], [3.0717], [-1.5502], [-11.1862], [4.3566], [3.1572], [3.0276], [-1.4563], [-3.8538]], device='cuda:0'),
    torch.tensor([[-4.2615], [2.3747], [0.5351], [-10.5619], [4.8812], [0.1865], [2.0936], [-2.8460], [-2.3292]], device='cuda:0'),
    torch.tensor([[-4.7652], [3.5892], [-1.7800], [-12.1459], [4.8107], [2.4259], [5.1920], [-1.3982], [-3.8143]], device='cuda:0'),
    torch.tensor([[-4.4486e+00], [3.3668e+00], [5.1957e-03], [-9.8446e+00], [5.8042e+00], [1.7155e+00], [2.7282e+00], [-2.7765e+00], [-4.0970e+00]], device='cuda:0'),
    torch.tensor([[-4.3635], [3.0465], [-0.8389], [-10.5891], [4.6809], [3.1491], [3.3109], [-1.7173], [-4.6558]], device='cuda:0'),
    torch.tensor([[-4.6581], [3.1631], [-1.2883], [-11.0421], [4.5849], [2.9624], [3.6367], [-1.4899], [-4.4056]], device='cuda:0'),
    torch.tensor([[-4.5716], [3.2033], [-0.5147], [-10.2341], [5.5456], [2.2363], [2.8975], [-2.4116], [-3.8914]], device='cuda:0'),
    torch.tensor([[-4.2553], [2.8816], [-1.0501], [-10.9533], [3.4751], [3.0859], [2.5267], [-1.0908], [-4.4560]], device='cuda:0'),
    torch.tensor([[-4.4358], [3.3527], [-1.5207], [-11.2892], [4.7586], [2.9581], [3.2805], [-1.5436], [-3.9101]], device='cuda:0'),
    torch.tensor([[-3.9878], [3.2942], [-0.7462], [-10.6730], [5.1768], [3.0448], [2.3471], [-2.1827], [-3.8196]], device='cuda:0'),
    torch.tensor([[-3.9731], [2.7229], [-0.9481], [-10.5722], [3.4632], [3.4091], [1.4075], [-1.0466], [-4.2295]], device='cuda:0'),
    torch.tensor([[-4.4571], [3.5017], [-1.5378], [-11.6917], [4.4535], [2.9622], [4.3060], [-1.2373], [-4.1516]], device='cuda:0'),
    torch.tensor([[-4.4274], [4.4553], [-1.0762], [-13.3917], [4.3613], [-0.5651], [7.0825], [-0.6507], [-3.1150]], device='cuda:0'),
    torch.tensor([[-4.6586], [3.9031], [-1.9695], [-13.0076], [4.5627], [1.8667], [6.1798], [-0.9946], [-3.2242]], device='cuda:0'),
    torch.tensor([[-4.5208], [2.9070], [-1.2978], [-10.8709], [4.6973], [3.0746], [3.3353], [-1.8410], [-4.0810]], device='cuda:0'),
    torch.tensor([[-5.0333], [13.6979], [-3.8776], [-16.2662], [-1.1800], [10.5959], [5.5816], [-9.4204], [-8.0341]], device='cuda:0')
]

lgd_10d_swd = [np.float32(0.74982035), np.float32(0.40983894), np.float32(0.52998656),
               np.float32(0.76115566), np.float32(0.6997979), np.float32(0.5090483),
               np.float32(1.758918), np.float32(9.690183), np.float32(0.5418884),
               np.float32(0.67938745), np.float32(0.6880052), np.float32(0.5175528),
               np.float32(0.64725816), np.float32(0.7220188), np.float32(0.68135756),
               np.float32(0.66077816), np.float32(0.75528085), np.float32(0.6929233),
               np.float32(0.77296734), np.float32(0.7919495), np.float32(0.55942345),
               np.float32(0.37500182), np.float32(0.3075858), np.float32(0.7083479),
               np.float32(0.6930528)]

# LGD-CM
lgdcm_10d_xt = lgd_10d_xt  # Same as LGD based on your data structure

lgdcm_10d_swd = [np.float32(0.44006035), np.float32(0.4889687), np.float32(0.49983352),
                 np.float32(0.5268694), np.float32(0.51425594), np.float32(0.48321083),
                 np.float32(0.44367015), np.float32(0.75774175), np.float32(0.48766297),
                 np.float32(0.7965871), np.float32(0.42425343), np.float32(0.51760906),
                 np.float32(0.52745205), np.float32(0.54563177), np.float32(0.3377344),
                 np.float32(0.49511847), np.float32(0.4640403), np.float32(0.43583283),
                 np.float32(0.45292166), np.float32(0.5626694), np.float32(0.3826994),
                 np.float32(0.43780228), np.float32(0.48712626), np.float32(0.47876036),
                 np.float32(0.6999929)]

# D-FLOW
dflow_10d_xt = [
    torch.tensor([[15.1364], [-14.6969], [10.1000], [-2.9796], [-1.1665], [1.7796], [4.7004], [14.5525], [2.8453]], device='cuda:0'),
    torch.tensor([[-4.5550], [-8.9457], [1.0522], [-7.4888], [6.1504], [-9.3776], [8.0880], [6.5449], [-3.1814]], device='cuda:0'),
    torch.tensor([[-4.8312], [4.2170], [1.2129], [-10.8156], [2.9060], [-1.6161], [1.7237], [-1.1276], [-3.6155]], device='cuda:0'),
    torch.tensor([[-4.1806], [-0.2676], [-0.8661], [-8.5943], [3.3199], [5.5889], [0.8588], [-2.0139], [-5.9333]], device='cuda:0'),
    torch.tensor([[-2.1765], [1.2757], [-4.1146], [-4.7380], [2.5796], [3.5778], [0.3942], [-4.7020], [-5.6733]], device='cuda:0'),
    torch.tensor([[-5.2292], [-0.6350], [-2.2106], [-5.5144], [2.1447], [0.5025], [-5.2368], [-4.0202], [-9.1066]], device='cuda:0'),
    torch.tensor([[-4.0244], [6.9860], [-6.0085], [-14.0192], [6.3393], [-3.0298], [3.5630], [-2.1222], [-3.6006]], device='cuda:0'),
    torch.tensor([[-2.5074], [6.7685], [2.6231], [-11.4778], [4.4789], [-1.7530], [1.6901], [-1.0395], [-3.8399]], device='cuda:0'),
    torch.tensor([[-5.8665], [2.1905], [-0.1560], [-6.9809], [4.8925], [-2.7704], [4.8607], [-1.6667], [-6.4696]], device='cuda:0'),
    torch.tensor([[-4.5580], [4.2377], [1.5612], [-7.7885], [8.6482], [2.2850], [5.3374], [-3.3645], [-7.6289]], device='cuda:0'),
    torch.tensor([[-7.2750], [-3.7065], [-1.0415], [-5.3033], [3.0480], [5.1354], [-3.3484], [-8.0806], [-6.6121]], device='cuda:0'),
    torch.tensor([[16.7732], [-9.5251], [10.4271], [-6.5255], [0.5274], [0.1953], [3.9020], [19.5769], [3.0107]], device='cuda:0'),
    torch.tensor([[-0.5152], [-2.2428], [-5.0697], [-4.9060], [3.2468], [2.9251], [-4.5448], [-5.9045], [-7.6906]], device='cuda:0'),
    torch.tensor([[-2.2903], [4.5033], [-1.0291], [-14.5529], [4.5644], [-3.9158], [4.5424], [-3.5905], [-3.3264]], device='cuda:0'),
    torch.tensor([[0.6198], [6.2777], [16.0558], [-8.7093], [6.8647], [3.0811], [-2.2709], [3.3069], [-3.0213]], device='cuda:0'),
    torch.tensor([[-7.2864], [4.7787], [-3.6744], [-14.0985], [6.0985], [-2.9352], [3.9123], [-2.4610], [-6.9394]], device='cuda:0'),
    torch.tensor([[-6.2832], [7.1714], [-5.4000], [-13.3876], [4.6205], [6.8525], [1.0921], [-5.0220], [-0.8513]], device='cuda:0'),
    torch.tensor([[14.8105], [-12.4020], [13.9695], [-12.6580], [4.0179], [-0.5282], [5.3646], [18.8380], [5.7362]], device='cuda:0'),
    torch.tensor([[-4.9629], [0.0839], [-2.8888], [-11.3211], [-0.0472], [7.8955], [3.1710], [-0.4442], [-5.2881]], device='cuda:0'),
    torch.tensor([[-6.5571], [-3.9801], [-2.9979], [2.7893], [1.9042], [5.9392], [2.6060], [-7.8307], [-8.0964]], device='cuda:0'),
    torch.tensor([[-6.4434], [0.1828], [-3.9746], [-2.9824], [1.3679], [1.1915], [-3.1139], [-2.0613], [-9.1504]], device='cuda:0'),
    torch.tensor([[-4.4559], [2.1696], [1.4034], [-14.0675], [1.5046], [-6.5448], [3.6467], [0.0752], [1.4799]], device='cuda:0'),
    torch.tensor([[17.3481], [-13.9621], [16.7247], [-3.9847], [3.9648], [0.4663], [11.6144], [20.1423], [4.2214]], device='cuda:0'),
    torch.tensor([[16.1395], [-13.1864], [14.0887], [-9.0579], [6.1183], [-3.2012], [7.9253], [21.5263], [3.4212]], device='cuda:0'),
    torch.tensor([[-3.4913], [7.7026], [-3.7737], [-13.2496], [9.7081], [-0.5754], [5.3364], [-2.5977], [-3.9745]], device='cuda:0')
]

dflow_10d_swd = [np.float32(0.646324), np.float32(0.73372024), np.float32(0.58874804),
                 np.float32(0.5308777), np.float32(0.68092513), np.float32(0.586321),
                 np.float32(0.4670589), np.float32(0.99234486), np.float32(0.5590603),
                 np.float32(0.22049233), np.float32(0.59751374), np.float32(0.7840888),
                 np.float32(0.6419953), np.float32(0.6323571), np.float32(0.54833376),
                 np.float32(0.45818716), np.float32(0.6844877), np.float32(0.77478415),
                 np.float32(0.74395967), np.float32(0.61202544), np.float32(0.73839754),
                 np.float32(0.71493554), np.float32(0.6965006), np.float32(0.7854252),
                 np.float32(0.46598917)]
# END CHANGED

# CHANGED: Define calculate_l2_distances_10d function
# OLD: Function didn't exist
def calculate_l2_distances_10d(xt_list, swd_list, optimal, name):
    """Calculate L2 distances for 10D case"""
    # Convert tensors to numpy arrays
    xt_arrays = []
    for x in xt_list:
        if isinstance(x, torch.Tensor):
            xt_arrays.append(x.cpu().detach().squeeze().numpy())
        else:
            xt_arrays.append(np.array(x).squeeze())

    optimal_np = optimal.cpu().numpy() if isinstance(optimal, torch.Tensor) else np.array(optimal)

    # Calculate L2 distances
    l2_distances = [np.linalg.norm(x - optimal_np) for x in xt_arrays]

    # Get top 10 indices based on SWD
    top10_indices = np.argsort(swd_list)[:10]

    # All attempts
    all_mean = np.mean(l2_distances)
    all_std = np.std(l2_distances)

    # Top 10 attempts
    top10_l2 = [l2_distances[i] for i in top10_indices]
    top10_mean = np.mean(top10_l2)
    top10_std = np.std(top10_l2)

    print(f"\n{name}:")
    print(f"  All attempts: Mean L2 = {all_mean:.4f}, Std L2 = {all_std:.4f}")
    print(f"  Top 10 SWD:   Mean L2 = {top10_mean:.4f}, Std L2 = {top10_std:.4f}")

    return {
        'all_mean': all_mean,
        'all_std': all_std,
        'top10_mean': top10_mean,
        'top10_std': top10_std
    }
# END CHANGED

print("\n" + "="*60)
print("10D CONDITIONAL 9D - L2 DISTANCES TO OPTIMAL")
print("="*60)

results_10d = {}
results_10d['SUPERPOSITION'] = calculate_l2_distances_10d(superposition_10d_xt, superposition_10d_swd, optimal_10d, "SUPERPOSITION")
results_10d['LGD'] = calculate_l2_distances_10d(lgd_10d_xt, lgd_10d_swd, optimal_10d, "LGD")
results_10d['LGD-CM'] = calculate_l2_distances_10d(lgdcm_10d_xt, lgdcm_10d_swd, optimal_10d, "LGD-CM")
results_10d['D-FLOW'] = calculate_l2_distances_10d(dflow_10d_xt, dflow_10d_swd, optimal_10d, "D-FLOW")